# V40 — Within-category share of wallet, free common pi, household bootstrap

Derived from the attached V39. **Same input Parquet path, sample rules, five categories, covariates, free common initial probability, and household cluster bootstrap.** Run all cells after checking `INPUT_PATH`. Dependencies: `numpy pandas scipy duckdb pyarrow tqdm threadpoolctl`.

## Outcome and denominator
For each trip with positive included category expenditure, use component expenditure divided by **total included expenditure in that category on that trip**. Each such trip contributes one unit of fractional composition mass, regardless of dollars spent. This is not healthy spending divided by all food spending.

| Category | Preferred numerator | Category denominator |
|---|---|---|
| FV | Fresh FV | Fresh + non-fresh FV |
| Yogurt | Fat-free yogurt | Fat-free + low-fat + other yogurt |
| Cereal | Low- + medium-sugar cereal | Low + medium + high-sugar classified cereal |
| Snack | Healthy snacks | Healthy + unhealthy snacks |
| Beverage including milk | Healthy beverage + skimmed, low-fat and other milk | Healthy + unhealthy beverage + all three milk groups |

Milk remains inside beverages, exactly as in V39; there is no separate milk emission and no new product reclassification. Yogurt remains fat-free only in the preferred numerator. For example, CAD 2 healthier + CAD 8 less-healthy spending contributes a share of **0.20**, even if the old priority code is 1. Unclassified cereal is outside the denominator.

## Fractional objective: interpretation matters
For binary categories, the conditional contribution is
$$q\log g_s+(1-q)\log(1-g_s),\qquad g_s=\operatorname{logit}^{-1}(\alpha+\delta s+X'\beta).$$
Cereal now uses the same single binary fractional gate as the other categories:
$$q_C=\frac{spend_{low}+spend_{medium}}{spend_{low}+spend_{medium}+spend_{high}}.$$
The complementary share is high-sugar cereal. **Healthy** is the operational low-or-medium-sugar group; it is not a claim about overall nutritional quality. Exact endpoints 0 and 1 are retained, with no epsilon replacement. Original Parquet codes remain 0/1/2/3; the read-only view maps 1 and 2 to the healthy group, 3 to high sugar, and 0 to nonpurchase. Purchase incidence is unchanged. All fractions are calculated from spending, never from priority codes.

Annual sums of these trip fractions are exact sufficient statistics for this objective. Forward–backward and safeguarded EM optimize a **latent-state fractional pseudo-likelihood**; the fractional kernels are not normalized densities for continuous shares. This is a descriptive fractional extension of V39, not a fully specified continuous-outcome HMM. The state weights are pseudo-posterior weights. A proper generative share HMM would require a specified density including endpoint masses and different estimation code. Household bootstrap estimates sampling variability of this estimator; it does not establish latent-state identification or correct model specification.

State-independent purchase-incidence logits still use V39's observed category codes and `food_spend/10`. Incidence remains separable. Conditional shares are observed only when category spending is positive. Purchases with zero expenditure retain their incidence observation but contribute no composition information; they are audited. Positive expenditure with a nonpurchase code is rejected. Missing/nonfinite spending and negatives below -1e-8 are rejected; negative floating-point noise in [-1e-8,0) is set to zero and counted. There is no substantive clipping or deletion of trips.

**Fit reporting:** `log_pseudo_likelihood` is reported; AIC, BIC and their deltas are deliberately unavailable. Neither conventional information-criterion nor likelihood-ratio inference is justified by this fractional kernel. Do not compare its objective to V39's discrete likelihood. Internal `log_likelihood` attributes are retained only for optimizer compatibility.

## Preserved structure and inference
- State $s\in\{1,2\}$ at household-year level; trip-level emissions aggregated within year.
- Choice controls: children under 18, income in $10,000, household size and age.
- Transition logits: new child, marriage, divorce/separation, moving, getting a job and retirement.
- No-switch comparison fixes transitions to the identity matrix and refits choices and common pi.
- Each model estimates its own common $\pi=P(S_{i,t_0}=2)$ from first-observed-year state weights. The legacy 2007 share is ignored.
- Parameter counts remain **55 proposed / 41 no-switch**.
- One-year households are retained. Observed-year gaps follow V39's `observed_step` convention; set `YEAR_GAP_POLICY='error'` to reject gaps.
- Entire household histories are resampled with replacement. Every draw refits incidence, shares, transitions and pi. Multiplicity weights reproduce complete-history duplication without trip expansion.
- Default 200 draws; 1,000 is a useful final-reporting starting point, subject to convergence and Monte Carlo stability. Bootstrap SDs, full covariance and percentile intervals are used. Wald p-values/stars use bootstrap SEs; no inverse-Hessian inference.
- Failed or unsuitable fits remain logged; full-data inference needs at least max(100,80% of requested draws) accepted and a suitable original fit. Synthetic mode is only a software check.
- FV orients full-sample states; draws are matched to their own reference fit. Inspect all profiles before calling states healthy/unhealthy.

## Outputs
`11_posterior_healthy_wallet_shares.csv` contains posterior-weighted **mean trip shares**, matching the modeled outcome. `12_posterior_healthy_pooled_dollar_shares.csv` contains posterior-weighted preferred dollars / category dollars, a separate dollar-weighted descriptive measure. `04_posterior_purchase_profiles.csv` includes both, purchase incidence, and category spending / all-food spending. Profile uncertainty is recomputed through household bootstrap.

`share_input_audit.csv` records zero-spend purchases, mixed baskets, endpoint/interior shares and tiny-negative corrections. `annual_sufficient_statistics.csv` labels fractional masses `_share_sum`; `_n` fields remain actual trip counts. Results and checkpoints go to **V40_Share_of_Wallet_LowMediumHealthy_Results**, separate from V39. The input file is read only.

This revision removes the medium-given-not-low cereal gate (six coefficients). It uses a new result folder so earlier three-group cereal checkpoints cannot be mixed with this specification.


## Validation
Cells are delivered unexecuted. Synthetic checks verify the combined low-plus-medium cereal numerator, original 0/1/2/3 code mapping, unchanged purchase incidence, one cereal gate, trip-versus-annual objective equivalence, both model fits, household bootstrap, checkpoint resume and outputs. Expected parameter counts are 55 proposed / 41 no-switch. No empirical estimates are supplied because the purchase-data Parquet is not attached.


In [ ]:
import os

# Numerical-library thread settings MUST run before importing NumPy/SciPy.
# HMM_NUM_THREADS overrides the conservative default of min(4, detected CPUs).
_thread_text = os.environ.get("HMM_NUM_THREADS")
if _thread_text is None:
    HMM_NUM_THREADS = min(4, os.cpu_count() or 1)
else:
    try:
        HMM_NUM_THREADS = int(_thread_text)
    except (TypeError, ValueError) as exc:
        raise ValueError("HMM_NUM_THREADS must be an integer >= 1.") from exc
    if HMM_NUM_THREADS < 1:
        raise ValueError("HMM_NUM_THREADS must be an integer >= 1.")
for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_variable] = str(HMM_NUM_THREADS)
print(f"Requested numerical threads = {HMM_NUM_THREADS}")
print("Thread settings take effect only before NumPy/SciPy import; after changing them, Restart Kernel and Run All.")

import gc
from copy import copy
import math
import time
import warnings
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit, logsumexp
from scipy.stats import norm
try:
    from tqdm.auto import tqdm
except ImportError:
    class tqdm:
        """Minimal progress fallback used only when tqdm is unavailable."""
        def __init__(self, iterable=None, total=None, desc="", **kwargs):
            self.iterable = iterable; self.total = total; self.desc = desc; self.n = 0
        def __enter__(self):
            print(self.desc); return self
        def __exit__(self, exc_type, exc, tb):
            return False
        def __iter__(self):
            for item in self.iterable:
                yield item; self.n += 1
        def update(self, value=1):
            self.n += value
        def set_description_str(self, text):
            self.desc = text; print(text)
        def set_postfix_str(self, text):
            pass
        def set_postfix(self, **kwargs):
            pass


import duckdb
import json
INPUT_PATH = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data\processed\hmm_input\hmm_estimation_panel.parquet")
OUTPUT_DIR = INPUT_PATH.parent / 'V40_Share_of_Wallet_LowMediumHealthy_Results'
RUN_MODE = 'full_estimation'  # 'full_estimation', 'validate_only', or 'synthetic'
MAX_EM_ITERS = 150
SYNTHETIC_EM_ITERS = 40
MIN_OBSERVED_YEARS_PER_HOUSEHOLD = 1  # No extra three-year household filter.
YEAR_GAP_POLICY = 'observed_step'  # 'error' rejects calendar-year gaps.
DUCKDB_MEMORY_LIMIT = '4GB'
CHECK_DUPLICATE_TRIPS = True
TOL=1e-3
EM_REL_TOL=1e-10
EM_STABLE_ITERS=3
EM_PRINT_EVERY=5
OPT_FTOL=1e-14
OPT_GTOL=1e-8
OPT_NEWTON_STEPS=20
OPT_FAST_NEWTON_STEPS=8
SEED=20261030
CEREAL_CHOICE_WEIGHT=1.0
# The legacy initial_fresh_fv_share_2007 input column is ignored.
INITIAL_PI_START = 0.5
PI_EPS = 1e-10  # numerical boundary guard, not an informative prior
INCIDENCE_CACHE_RAM_MB = 256  # construction-peak bound; larger caches stay on disk
BOOTSTRAP_WORKERS = 'auto'  # calibrates 1 vs 2 workers using actual bootstrap draws
BOOTSTRAP_RESUME = True     # verifies source contents, numerical code and settings
BOOTSTRAP_EXPORT_EVERY = 10 # atomic per-draw checkpoints always save immediately
BOOTSTRAP_REPS = 200
BOOTSTRAP_SEED = 20261114
BOOTSTRAP_MIN_SUCCESS = 100
BOOTSTRAP_MIN_SUCCESS_FRACTION = 0.80
BOOTSTRAP_MAX_EM_ITERS = 300
SYNTHETIC_BOOTSTRAP_REPS = 5
BOOTSTRAP_BOUNDARY_TOL = 1e-7

TRANSITION_EVENT_COLUMNS=['event_new_child','getting_married','getting_divorce_or_separation','moving','getting_job','retirement']
TRANSITION_X_COLS=['intercept',*TRANSITION_EVENT_COLUMNS]
GATE_COVARIATE_COLS=['K_1','income10k_imputed','family_size_numeric','age']
GATE_X_COLS=['intercept','state_num',*GATE_COVARIATE_COLS]
CATEGORY_CONFIG=OrderedDict()
_spec=[
 ('fv',['fresh','nonfresh'],['fresh_spending','non_fresh_spending']),
 ('yogurt',['fat_free','other_yogurt'],['skimmed_yogurt_spending','low_fat_yogurt_spending','other_yogurt_spending']),
 ('cereal',['healthy_low_medium','high_sugar'],['low_sugar_cereal_spending','medium_sugar_cereal_spending','high_sugar_cereal_spending']),
 ('snack',['healthy','unhealthy'],['healthy_snack_spending','unhealthy_snack_spending']),
 ('beverage',['healthier','less_healthy'],['healthy_beverage_spending','unhealthy_beverage_spending',
  'skimmed_milk_spending','low_fat_milk_spending','other_milk_spending'])]
for c,labels,spending in _spec:
 gates=[dict(key=c+'_'+labels[0],prefix=c+'_'+labels[0]+'_gate',eligible_codes=tuple(range(1,len(labels)+1)),target_code=1)]
 CATEGORY_CONFIG[c]=dict(y_col=c+'_y',outcomes=OrderedDict(enumerate(['no_classified_purchase' if c=='cereal' else 'no_purchase',*labels])),gates=gates,spending=spending,choice_weight=1.0)
CATEGORY_NAMES=list(CATEGORY_CONFIG)
assert CATEGORY_NAMES == ['fv','yogurt','cereal','snack','beverage']
assert 'milk' not in CATEGORY_CONFIG
GATE_KEYS=[g['key'] for c in CATEGORY_CONFIG.values() for g in c['gates']]
np.random.seed(SEED)

# Mutually exclusive expenditure groups, in the positive-outcome order above.
SPENDING_NEGATIVE_TOL = 1e-8
SHARE_COMPONENTS = {
 'fv': [['fresh_spending'], ['non_fresh_spending']],
 'yogurt': [['skimmed_yogurt_spending'], ['low_fat_yogurt_spending','other_yogurt_spending']],
 'cereal': [['low_sugar_cereal_spending','medium_sugar_cereal_spending'], ['high_sugar_cereal_spending']],
 'snack': [['healthy_snack_spending'], ['unhealthy_snack_spending']],
 'beverage': [['healthy_beverage_spending','skimmed_milk_spending','low_fat_milk_spending','other_milk_spending'], ['unhealthy_beverage_spending']],
}
for _c, _groups in SHARE_COMPONENTS.items():
    _flat = [v for group in _groups for v in group]
    assert len(_flat)==len(set(_flat)) and set(_flat)==set(CATEGORY_CONFIG[_c]['spending'])


## 1. Numerical engine
Analytic gradients and exact small-block Hessians support damped Newton steps with L-BFGS fallback. Scaling affects only optimizer coordinates. The objective has no ridge penalty. Original gradient/EM tolerances, stable-iteration requirements, and downhill-step rejection are retained. Hessians are used for numerical optimization and rank diagnostics only; reported inference uses household bootstrap variances.


In [ ]:
@dataclass
class GateBlock:
    """Annual fractional masses mapped to one shared unique-covariate design."""

    key: str
    category: str
    prefix: str
    x_base: np.ndarray
    target: np.ndarray
    hy_pos: np.ndarray
    eligible_rows: float
    choice_weight: float
    success_count: np.ndarray
    failure_count: np.ndarray
    design_pos: np.ndarray


@dataclass
class ModelArrays:
    """NumPy model data with cached household-year and gate structures."""

    purchase: Dict[str, np.ndarray]
    outcome: Dict[str, np.ndarray]
    spend: Dict[str, np.ndarray]
    food_total_spend: np.ndarray
    food_total_spend_10: np.ndarray
    gate_covariates: Dict[str, np.ndarray]
    gate_blocks: Dict[str, GateBlock]
    trip_hy_pos: np.ndarray
    household_year: pd.DataFrame
    hy_key: pd.MultiIndex
    hh_blocks: Dict[str, Dict[str, np.ndarray]]
    households: List[str]
    years_by_household: Dict[str, List[int]]
    hh_starts: np.ndarray
    hh_ends: np.ndarray
    transition_x: np.ndarray
    transition_curr_pos: np.ndarray
    transition_row_of_hy: np.ndarray


@dataclass
class PreparedData:
    """Annual sufficient statistics and model buffers.

    With RETAIN_TRIP_DATA=False, df is a count-only RangeIndex shell carrying
    the trip count and sorting audit. Raw trip buffers are released after the
    incidence fits; all subsequent EM operations use annual/design counts.
    """

    df: pd.DataFrame
    hy: pd.DataFrame
    arrays: ModelArrays
    annual_stats: pd.DataFrame


@dataclass
class EstimationResult:
    """Fitted model blocks and final optimizer diagnostics."""

    prepared: PreparedData
    incidence_models: Dict[str, Any]
    gate_params: Dict[str, Any]
    beta_from1: np.ndarray
    beta_from2: np.ndarray
    pi_state2: float
    gamma_df: pd.DataFrame
    history_df: pd.DataFrame
    initial_state_table: pd.DataFrame
    transition_table: pd.DataFrame
    log_likelihood: float
    transition_fits: Optional[Dict[str, Dict[str, Any]]] = None
    optimizer_diagnostics_df: Optional[pd.DataFrame] = None


def safe_log(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Take a stable logarithm of probabilities."""
    return np.log(np.clip(x, eps, 1.0))


def sigmoid(x: np.ndarray) -> np.ndarray:
    """Stable logistic transform."""
    return expit(x)


def significance_stars(p_value: float) -> str:
    """Return conventional significance stars."""
    if not np.isfinite(p_value):
        return ""
    if p_value < 0.01:
        return "***"
    if p_value < 0.05:
        return "**"
    if p_value < 0.10:
        return "*"
    return ""


def ensure_columns(df: pd.DataFrame, required_cols: Sequence[str]) -> None:
    """Raise a clear error when required columns are missing."""
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def first_nonmissing(series: pd.Series) -> float:
    """Return the first observed value, preserving missingness when all values are missing."""
    observed = series.dropna()
    return observed.iloc[0] if len(observed) else np.nan


def _optimizer_diagnostics(
    result: Any,
    block_name: str,
    objective_value: float,
    gradient: np.ndarray,
    n_observations: int,
    effective_weight: float,
    covariance_success: bool,
    solver: str,
) -> Dict[str, Any]:
    return {
        "scaled_gradient_inf": float(getattr(result, 'scaled_gradient_inf', np.nan)),
        "initial_objective": float(getattr(result, 'initial_objective', np.nan)),
        "objective_improvement": float(getattr(result, 'objective_improvement', np.nan)),
        "reverted_to_start": bool(getattr(result, 'reverted_to_start', False)),
        "lbfgs_message": str(getattr(result, 'lbfgs_message', '')),
        "newton_refinements": int(getattr(result, 'newton_refinements', 0)),
        "model_block": block_name,
        "solver": solver,
        "optimizer_success": bool(result.success),
        "status": int(result.status),
        "message": str(result.message),
        "iterations": int(getattr(result, "nit", -1)),
        "function_evaluations": int(getattr(result, "nfev", -1)),
        "gradient_evaluations": int(getattr(result, "njev", -1)),
        "final_objective": float(objective_value),
        "gradient_norm": float(np.linalg.norm(gradient)),
        "n_observations": int(n_observations),
        "effective_weight": float(effective_weight),
        "covariance_hessian_success": bool(covariance_success),
    }


def _finish_logit_fit(
    result: Any,
    hessian_function: Any,
    gradient_function: Any,
    param_names: Sequence[str],
    block_name: str,
    objective_function: Any,
    n_observations: int,
    effective_weight: float,
    compute_se: bool,
    solver: str,
) -> Dict[str, Any]:
    beta = np.asarray(result.x, dtype=np.float64)
    # Block Hessians are used for numerical optimization/rank diagnostics only.
    # No inverse-Hessian variance path is permitted, even when called directly.
    if compute_se:
        raise ValueError('Block-Hessian inference is disabled. Run the household bootstrap for variance and SEs.')
    covariance_success = False
    hessian = np.full((len(beta), len(beta)), np.nan)
    covariance = np.full((len(beta), len(beta)), np.nan)
    standard_error = np.full(len(beta), np.nan)

    gradient = np.asarray(gradient_function(beta), dtype=np.float64)
    objective_value = float(objective_function(beta))
    with np.errstate(divide="ignore", invalid="ignore"):
        z_value = beta / standard_error
        p_value = 2.0 * (1.0 - norm.cdf(np.abs(z_value)))
    table = pd.DataFrame({
        "parameter": list(param_names),
        "estimate": beta,
        "std_error": standard_error,
        "z_value": z_value,
        "p_value": p_value,
    })
    table["significance_stars"] = table["p_value"].map(significance_stars)
    diagnostics = _optimizer_diagnostics(
        result, block_name, objective_value, gradient, n_observations,
        effective_weight, covariance_success, solver,
    )
    if not result.success and not getattr(result,'evaluation_only',False):
        warnings.warn(
            f"Optimizer did not report success for {block_name}: "
            f"status={result.status}; message={result.message}",
            RuntimeWarning,
        )
    return {
        "_hessian_function": hessian_function,
        "result": result,
        "coef": beta,
        "hessian": hessian,
        "cov": covariance,
        "table": table,
        "nll": objective_value,
        "diagnostics": diagnostics,
    }


def _cached_value_gradient(callback):
    """Reuse one matrix product/sigmoid pass for both L-BFGS-B callbacks."""
    last_beta = None
    last_result = None
    def evaluate(beta):
        nonlocal last_beta, last_result
        if last_beta is None or not np.array_equal(beta, last_beta):
            last_beta = np.array(beta, copy=True)
            last_result = callback(beta)
        return last_result
    return evaluate


def _minimize_logit(evaluate, hess, initial, effective_weight, maxiter):
    """Scaled L-BFGS with gradient verification and damped Newton refinement.

    Scaling changes only optimizer coordinates, never data or likelihood.
    The starting coefficients are always a candidate, so a block M-step
    cannot decrease its expected complete-data log likelihood.
    """
    from scipy.optimize import OptimizeResult
    weight = max(float(effective_weight), 1.0)
    initial = np.array(initial, dtype=float, copy=True)
    curvature = np.maximum(np.diag(hess(np.zeros_like(initial))) / weight, 0.0)
    scale = np.sqrt(curvature)
    scale[scale < 1e-8] = 1.0
    calls = 0

    def objective(theta):
        nonlocal calls
        calls += 1
        value, gradient = evaluate(theta / scale)
        return value / weight, gradient / (weight * scale)

    theta0 = initial * scale
    f0, g0 = objective(theta0)
    best_theta, best_f = theta0.copy(), f0
    raw_message = 'Newton-first step; L-BFGS fallback not required'
    nit = 0
    # Six-/seven-parameter convex blocks: a few exact Newton steps normally
    # reach the same gradient tolerance much sooner than L-BFGS warm starts.
    # Damping, monotone line search and L-BFGS fallback preserve safeguards.
    fast_refinements = 0
    for _ in range(min(OPT_FAST_NEWTON_STEPS, maxiter)):
        value, grad = objective(best_theta)
        if np.linalg.norm(grad, ord=np.inf) <= OPT_GTOL:
            break
        H = hess(best_theta / scale) / weight / scale[:, None] / scale[None, :]
        jitter = 1e-10 * max(1.0, float(np.max(np.diag(H))))
        try:
            direction = -np.linalg.solve(H + jitter*np.eye(len(H)), grad)
        except np.linalg.LinAlgError:
            direction = -np.linalg.pinv(H) @ grad
        slope = float(grad @ direction)
        if not np.isfinite(direction).all() or slope >= 0:
            break
        accepted = False
        for backtrack in range(30):
            theta = best_theta + (0.5**backtrack)*direction
            candidate_f, _ = objective(theta)
            if np.isfinite(candidate_f) and candidate_f <= value + 1e-4*(0.5**backtrack)*slope:
                best_theta, best_f = theta, candidate_f
                fast_refinements += 1
                accepted = True
                break
        if not accepted:
            break
    _, fast_gradient = objective(best_theta)
    if maxiter > 0 and np.linalg.norm(fast_gradient, ord=np.inf) > OPT_GTOL:
        res = minimize(objective, best_theta, jac=True, method='L-BFGS-B', options={
            'maxiter': maxiter, 'maxfun': max(20000, 30 * maxiter),
            'ftol': OPT_FTOL, 'gtol': OPT_GTOL, 'maxls': 50, 'maxcor': 20,
        })
        raw_message, nit = str(res.message), int(res.nit)
        candidate_f, _ = objective(res.x)
        if np.isfinite(candidate_f) and candidate_f <= best_f:
            best_theta, best_f = res.x.copy(), candidate_f

    # A relative-function stop alone is insufficient on a massive likelihood.
    # Small dense Hessians (one logit block) make Newton polishing affordable.
    refinements = 0
    for _ in range(OPT_NEWTON_STEPS if maxiter > 0 else 0):
        value, grad = objective(best_theta)
        if np.linalg.norm(grad, ord=np.inf) <= OPT_GTOL:
            break
        H = hess(best_theta / scale) / weight / scale[:, None] / scale[None, :]
        jitter = 1e-10 * max(1.0, float(np.max(np.diag(H))))
        try:
            direction = -np.linalg.solve(H + jitter * np.eye(len(H)), grad)
        except np.linalg.LinAlgError:
            direction = -np.linalg.pinv(H) @ grad
        slope = float(grad @ direction)
        if not np.isfinite(direction).all() or slope >= 0:
            direction, slope = -grad, -float(grad @ grad)
        accepted = False
        for backtrack in range(40):
            step = 0.5 ** backtrack
            theta = best_theta + step * direction
            candidate_f, _ = objective(theta)
            if np.isfinite(candidate_f) and candidate_f <= value + 1e-4 * step * slope:
                best_theta, best_f = theta, candidate_f
                accepted = True
                refinements += 1
                break
        if not accepted:
            break
    # Recheck unscaled loss, preventing a rounding artifact in division from
    # accepting an objectively worse solution than the warm start.
    beta = best_theta / scale
    initial_value, _ = evaluate(initial)
    final_value, final_gradient = evaluate(beta)
    reverted = not np.isfinite(final_value) or final_value > initial_value
    if reverted:
        beta = initial
        final_value, final_gradient = evaluate(beta)
    scaled_gradient = final_gradient / (weight * scale)
    score = float(np.linalg.norm(scaled_gradient, ord=np.inf))
    success = bool(np.isfinite(score) and score <= OPT_GTOL)
    return OptimizeResult(
        x=beta, fun=final_value, jac=final_gradient, success=success, evaluation_only=(maxiter==0),
        status=0 if success else 2, nit=nit + refinements + fast_refinements, nfev=calls, njev=calls,
        message=('Scaled gradient tolerance met' if success else
                 'Scaled gradient tolerance NOT met; inspect this block'),
        scaled_gradient_inf=score, initial_objective=initial_value,
        objective_improvement=initial_value-final_value, reverted_to_start=reverted,
        lbfgs_message=raw_message, newton_refinements=refinements + fast_refinements,
    )




def fit_weighted_logit(
    X, y, sample_weight=None, start=None, param_names=None, maxiter=300,
    compute_se=False, block_name="weighted_logit",
):
    """Weighted Bernoulli likelihood, evaluated once per optimizer candidate."""
    y = np.asarray(y, dtype=np.float64)
    weight = np.ones_like(y) if sample_weight is None else np.asarray(sample_weight, dtype=np.float64)
    if np.any(np.isfinite(weight) & (weight < 0)):
        raise ValueError(f"{block_name}: negative weights are not allowed.")
    names = list(param_names) if param_names is not None else [f"b{i}" for i in range(X.shape[1])]
    return fit_binomial_count_logit(
        X, weight*y, weight*(1-y), start, names, block_name, maxiter, compute_se,
    )


def fit_compressed_gate_logit(
    X_base, y, weight_state1, weight_state2, start, param_names, block_name,
    maxiter=300, compute_se=False,
):
    """Compatibility wrapper for the original two-state weighted-trip API."""
    y = np.asarray(y, dtype=np.float64)
    w1 = np.asarray(weight_state1, dtype=np.float64)
    w2 = np.asarray(weight_state2, dtype=np.float64)
    return fit_gate_count_logit(
        X_base, w1*y, w1*(1-y), w2*y, w2*(1-y), start,
        param_names, block_name, maxiter, compute_se,
    )


def fit_gate_count_logit(
    X_base, success1, failure1, success2, failure2, start,
    param_names, block_name, maxiter=300, compute_se=False,
):
    """Two-state fractional Bernoulli objective on unique, unrounded covariates.

    Fractional masses include annual pseudo-posterior weights and category weight.
    This kernel is not a normalized continuous-share density.
    """
    X_base = np.asarray(X_base, dtype=np.float64)
    counts = np.column_stack([success1, failure1, success2, failure2]).astype(np.float64, copy=False)
    if np.any(np.isfinite(counts) & (counts < 0)):
        raise ValueError(f"{block_name}: negative expected counts are not allowed.")
    keep = np.isfinite(counts).all(axis=1) & (counts.sum(axis=1)>0) & np.isfinite(X_base).all(axis=1)
    if not keep.all():
        X_base, counts = X_base[keep], counts[keep]
    if not len(counts):
        raise ValueError(f"{block_name}: no valid observations remain.")
    s1, f1, s2, f2 = counts.T
    t1, t2 = s1+f1, s2+f2
    p_count = X_base.shape[1]+1
    base_positions = np.r_[0, np.arange(2, p_count)]
    initial = np.zeros(p_count) if start is None else np.asarray(start, dtype=np.float64)

    component_beta = None
    component_values = None
    def components(beta):
        nonlocal component_beta, component_values
        if component_beta is None or not np.array_equal(beta, component_beta):
            shared = X_base @ beta[base_positions]
            component_values = (shared, sigmoid(shared+beta[1]), sigmoid(shared+2*beta[1]))
            component_beta = np.array(beta, copy=True)
        return component_values

    def value_gradient(beta):
        shared, p1, p2 = components(beta)
        eta1, eta2 = shared + beta[1], shared + 2 * beta[1]
        # Stable Bernoulli loss, exactly consistent with the analytic gradient.
        value = (np.dot(s1, np.maximum(-eta1, 0)) + np.dot(f1, np.maximum(eta1, 0))
                 + np.dot(t1, np.log1p(np.exp(-np.abs(eta1))))
                 + np.dot(s2, np.maximum(-eta2, 0)) + np.dot(f2, np.maximum(eta2, 0))
                 + np.dot(t2, np.log1p(np.exp(-np.abs(eta2)))))
        r1, r2 = t1*p1-s1, t2*p2-s2
        out = np.empty(p_count)
        out[base_positions] = X_base.T @ (r1+r2)
        out[1] = np.sum(r1+2*r2)
        return float(value), out
    evaluate = _cached_value_gradient(value_gradient)

    def hess(beta):
        _, p1, p2 = components(beta)
        c1, c2 = t1*p1*(1-p1), t2*p2*(1-p2)
        out = np.empty((p_count, p_count))
        out[np.ix_(base_positions, base_positions)] = X_base.T @ (X_base*(c1+c2)[:, None])
        cross = X_base.T @ (c1+2*c2)
        out[base_positions, 1] = cross
        out[1, base_positions] = cross
        out[1, 1] = np.sum(c1+4*c2)
        return out

    result = _minimize_logit(evaluate, hess, initial, float(counts.sum()), maxiter)
    return _finish_logit_fit(
        result, hess, lambda b:evaluate(b)[1], param_names, block_name,
        lambda b:evaluate(b)[0], len(counts), float(counts.sum()), compute_se,
        "scaled_lbfgs_newton_gate_logit",
    )


def fit_binomial_count_logit(
    X, success, failure, start, param_names, block_name, maxiter=300, compute_se=False,
):
    """Original binomial-count objective with a joint loss/gradient callback."""
    X = np.asarray(X, dtype=np.float64)
    success, failure = np.asarray(success, dtype=np.float64), np.asarray(failure, dtype=np.float64)
    if np.any(np.isfinite(success) & (success<0)) or np.any(np.isfinite(failure) & (failure<0)):
        raise ValueError(f"{block_name}: negative expected counts are not allowed.")
    total = success+failure
    keep = np.isfinite(success) & np.isfinite(failure) & (total>0) & np.isfinite(X).all(axis=1)
    if not keep.all():
        X, success, failure, total = X[keep], success[keep], failure[keep], total[keep]
    if not len(total):
        raise ValueError(f"{block_name}: no positive expected counts remain.")
    initial = np.zeros(X.shape[1]) if start is None else np.asarray(start, dtype=np.float64)

    def value_gradient(beta):
        eta = X @ beta
        p = sigmoid(eta)
        value = np.sum(success*np.logaddexp(0, -eta) + failure*np.logaddexp(0, eta))
        return float(value), X.T @ (total*p-success)
    evaluate = _cached_value_gradient(value_gradient)
    def hess(beta):
        p = sigmoid(X @ beta)
        return X.T @ (X*(total*p*(1-p))[:, None])

    result = _minimize_logit(evaluate, hess, initial, float(total.sum()), maxiter)
    return _finish_logit_fit(
        result, hess, lambda b:evaluate(b)[1], param_names, block_name,
        lambda b:evaluate(b)[0], len(total), float(total.sum()), compute_se,
        "scaled_lbfgs_newton_binomial_logit",
    )


def finalize_diagnostics(fit):
    """Record block rank for diagnostics; bootstrap supplies all reported SEs."""
    if '_hessian_function' not in fit:
        return  # Already diagnosed; shared incidence fits are finalized once.
    hessian = np.asarray(fit.pop('_hessian_function')(fit['coef']), dtype=float)
    fit['diagnostics']['hessian_rank'] = int(np.linalg.matrix_rank(hessian))
    fit['diagnostics']['parameter_count'] = len(fit['coef'])
    fit['diagnostics']['covariance_hessian_success'] = False


def set_household_weights(prepared, weights=None):
    """Integer multiplicities represent resampling complete household sequences."""
    arrays = prepared.arrays
    H = len(arrays.hh_starts)
    w = np.ones(H, dtype=float) if weights is None else np.asarray(weights, dtype=float)
    if w.shape != (H,) or not np.isfinite(w).all() or (w < 0).any() or not np.equal(w, np.floor(w)).all() or w.sum() != H:
        raise ValueError('Require H nonnegative integer household multiplicities summing to H.')
    arrays._hh_weights = w.copy()
    arrays._hy_weights = np.repeat(w, arrays.hh_ends-arrays.hh_starts)




## 2. Spending validation and exact fractional sufficient statistics
The source is read only. Each positive-spend category trip contributes a vector summing to one. Annual sums preserve the trip objective exactly, including mixed cereal baskets. Incidence remains based on purchase codes. Zero-spend purchases supply incidence information only. A household-by-unrounded-food-spending cache supports incidence refits in every draw.


In [ ]:
def sql_name(name):
    return '"'+name.replace('"','""')+'"'


def register_combined_beverage_view(con, path):
    """Read original V30 outcomes; merge only in a SQL view, never edit Parquet.

    Existing milk category is assumed to contain ordinary milk only, as agreed
    for this first test. No UPC/flavor screening or sweetened-milk relabeling.
    Validate original milk, beverage and cereal codes before replacement; reject an
    invalid/missing code. Cereal 1/2 merge to healthy; 3 maps to high sugar.
    Nulls are errors, never non-purchases.
    """
    con.from_parquet(str(path)).create_view('raw_source')
    actual={r[0] for r in con.execute('DESCRIBE raw_source').fetchall()}
    missing={'milk_y','beverage_y','cereal_y'}-actual
    if missing:
        con.close()
        raise ValueError(f'Missing source outcome columns: {missing}')
    invalid=con.execute("""
        SELECT count(*) FROM raw_source
        WHERE milk_y IS NULL OR beverage_y IS NULL OR cereal_y IS NULL
           OR NOT isfinite(CAST(cereal_y AS DOUBLE)) OR cereal_y NOT IN (0,1,2,3)
           OR NOT isfinite(CAST(milk_y AS DOUBLE))
           OR NOT isfinite(CAST(beverage_y AS DOUBLE))
           OR milk_y NOT IN (0,1,2) OR beverage_y NOT IN (0,1,2)
    """).fetchone()[0]
    if invalid:
        con.close()
        raise ValueError(f'{invalid:,} invalid original milk/beverage/cereal outcome rows; fix upstream.')
    con.execute("""
        CREATE VIEW raw AS SELECT * REPLACE (
            CASE WHEN milk_y IN (1,2) OR beverage_y = 1 THEN 1
                 WHEN beverage_y = 2 THEN 2 ELSE 0 END AS beverage_y,
            CASE WHEN cereal_y IN (1,2) THEN 1
                 WHEN cereal_y=3 THEN 2 ELSE 0 END AS cereal_y
        ) FROM raw_source
    """)


def register_share_view(con):
    """Validate and construct shares without changing purchase-incidence codes."""
    spending=[v for cfg in CATEGORY_CONFIG.values() for v in cfg['spending']]
    bad=' OR '.join(f'{sql_name(v)} IS NULL OR NOT isfinite(CAST({sql_name(v)} AS DOUBLE)) OR CAST({sql_name(v)} AS DOUBLE)<{-SPENDING_NEGATIVE_TOL}' for v in spending)
    if con.execute('SELECT count(*) FROM raw WHERE '+bad).fetchone()[0]:
        raise ValueError('Missing, nonfinite, or materially negative component expenditure; fix upstream.')
    con.execute('CREATE VIEW clean_spending AS SELECT * REPLACE ('+
        ','.join(f'greatest(CAST({sql_name(v)} AS DOUBLE),0.0) AS {sql_name(v)}' for v in spending)+') FROM raw')
    fields=[]
    for c,groups in SHARE_COMPONENTS.items():
        fields.append('('+ '+'.join(sql_name(v) for group in groups for v in group)+f') AS {c}_category_spend')
        for k,group in enumerate(groups,1):
            fields.append('('+ '+'.join(sql_name(v) for v in group)+f') AS {c}_component_{k}_spend')
    con.execute('CREATE VIEW share_amounts AS SELECT *,'+','.join(fields)+' FROM clean_spending')
    inconsistent=' OR '.join(f'({c}_y=0 AND {c}_category_spend>0)' for c in CATEGORY_NAMES)
    if con.execute('SELECT count(*) FROM share_amounts WHERE '+inconsistent).fetchone()[0]:
        raise ValueError('Positive category spending with a nonpurchase code; fix upstream.')
    fields=[]
    for c,groups in SHARE_COMPONENTS.items():
        for k in range(1,len(groups)+1):
            fields.append(f'CASE WHEN {c}_category_spend>0 THEN {c}_component_{k}_spend/{c}_category_spend ELSE 0.0 END AS {c}_share_{k}')
    con.execute('CREATE VIEW share_trips AS SELECT *,'+','.join(fields)+' FROM share_amounts')


def prepare_parquet(path, scratch_dir):
    """Validate first, then aggregate; never silently drop malformed records."""
    con=duckdb.connect()
    con.execute(f"SET threads={HMM_NUM_THREADS}")
    con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
    scratch_dir=Path(scratch_dir); scratch_dir.mkdir(parents=True,exist_ok=True)
    con.execute("SET temp_directory='"+str(scratch_dir).replace("'","''")+"'")
    register_combined_beverage_view(con, path)
    controls=[*GATE_COVARIATE_COLS,*TRANSITION_EVENT_COLUMNS]
    spends=[s for cfg in CATEGORY_CONFIG.values() for s in cfg['spending']]
    required=['hhid','panel_year','trip_id','purchase_date','food_spend',*controls,*[c+'_y' for c in CATEGORY_NAMES],*spends]
    actual={r[0] for r in con.execute('DESCRIBE raw').fetchall()}
    if set(required)-actual: raise ValueError(f'Missing columns: {set(required)-actual}')
    bad=[f'{sql_name(c)} IS NULL' for c in required]
    numeric=['panel_year','food_spend',*controls,*spends,*[c+'_y' for c in CATEGORY_NAMES]]
    bad += [f'NOT isfinite(CAST({sql_name(c)} AS DOUBLE))' for c in numeric]
    bad += ['panel_year NOT BETWEEN 2008 AND 2017','panel_year != floor(panel_year)',"TRY_CAST(purchase_date AS DATE) IS NULL"]
    bad += [f'{c} NOT IN (0,1)' for c in [*TRANSITION_EVENT_COLUMNS,'K_1']]
    bad += [f'{c}_y NOT IN ({",".join(map(str,cfg["outcomes"]))})' for c,cfg in CATEGORY_CONFIG.items()]
    bad += ['family_size_numeric <= 0','age <= 0','age > 120','income10k_imputed < 0']
    n,b=con.execute('SELECT count(*), count(*) FILTER (WHERE '+ ' OR '.join(bad)+') FROM raw').fetchone()
    if b: raise ValueError(f'{b:,} invalid rows; fix upstream rather than silently filtering.')
    if not n: raise ValueError('Empty input Parquet.')
    if CHECK_DUPLICATE_TRIPS:
        duplicate=con.execute('SELECT 1 FROM raw GROUP BY hhid,panel_year,trip_id HAVING count(*)>1 LIMIT 1').fetchone()
        if duplicate: raise ValueError('Duplicate household-year-trip key.')
    checks=' OR '.join(f'min({c}) != max({c})' for c in controls)
    if con.execute(f'SELECT 1 FROM raw GROUP BY hhid,panel_year HAVING {checks} LIMIT 1').fetchone():
        raise ValueError('Demographics/events vary within a household-year.')
    con.execute(f'CREATE TEMP TABLE eligible AS SELECT hhid FROM raw GROUP BY hhid HAVING count(DISTINCT panel_year)>={MIN_OBSERVED_YEARS_PER_HOUSEHOLD}')
    register_share_view(con)
    con.execute('CREATE VIEW trips AS SELECT r.* FROM share_trips r SEMI JOIN eligible e USING(hhid)')
    share_audit=[]
    for c,cfg in CATEGORY_CONFIG.items():
        fields=["count(*) AS n_trips", f"count(*) FILTER (WHERE {c}_y>0) AS purchase_trips",
            f"count(*) FILTER (WHERE {c}_category_spend>0) AS positive_spend_trips",
            f"count(*) FILTER (WHERE {c}_y>0 AND {c}_category_spend=0) AS zero_spend_purchase_trips",
            f"count(*) FILTER (WHERE {c}_category_spend>0 AND {c}_share_1=0) AS preferred_share_zero_trips",
            f"count(*) FILTER (WHERE {c}_share_1=1) AS preferred_share_one_trips",
            f"count(*) FILTER (WHERE {c}_share_1>0 AND {c}_share_1<1) AS preferred_share_interior_trips"]
        nonzero='+'.join(f'CAST({c}_component_{k}_spend>0 AS INTEGER)' for k in range(1,len(SHARE_COMPONENTS[c])+1))
        fields.append(f'count(*) FILTER (WHERE ({nonzero})>1) AS mixed_component_trips')
        record=con.execute('SELECT '+','.join(fields)+' FROM trips').df().iloc[0].to_dict()
        corrected='+'.join(f'CAST({sql_name(v)}<0 AS BIGINT)' for v in cfg['spending'])
        record['tiny_negative_components_corrected']=con.execute('SELECT sum('+corrected+') FROM raw SEMI JOIN eligible USING(hhid)').fetchone()[0]
        record['category']=c; share_audit.append(record)
    share_audit=pd.DataFrame(share_audit)
    # Only nine combinations: cheap audit of the original codes and merged codes.
    merge_audit=con.execute("""
        SELECT r.milk_y AS original_milk_y, r.beverage_y AS original_beverage_y,
               CASE WHEN r.milk_y IN (1,2) OR r.beverage_y=1 THEN 1
                    WHEN r.beverage_y=2 THEN 2 ELSE 0 END AS merged_beverage_y,
               count(*) AS n_trips
        FROM raw_source r SEMI JOIN eligible e USING(hhid)
        GROUP BY 1,2,3 ORDER BY 1,2
    """).df()
    # SQL aliases retain the internal V29 household key to reuse its tested recursion.
    aggs=['CAST(hhid AS VARCHAR) AS household_code','CAST(panel_year AS INTEGER) AS panel_year','count(*) AS n_trips',
          'sum(CAST(food_spend AS DOUBLE)) AS food_total_spend_sum',
          'min(CAST(purchase_date AS DATE)) AS first_purchase_date','max(CAST(purchase_date AS DATE)) AS last_purchase_date']
    aggs += [f'first(CAST({c} AS DOUBLE)) AS {c}' for c in controls]
    for c,cfg in CATEGORY_CONFIG.items():
        aggs += [f'sum(CAST({c}_y>0 AS BIGINT)) AS {c}_purchase_n',
                 'sum('+ '+'.join(f'CAST({v} AS DOUBLE)' for v in cfg['spending'])+f') AS {c}_spend_sum']
        aggs += [f'sum(CAST({c}_y={k} AS BIGINT)) AS {c}_{v}_n' for k,v in cfg['outcomes'].items() if k]
        aggs += [f'sum(CAST({c}_category_spend>0 AS BIGINT)) AS {c}_share_observed_n']
        for k,v in cfg['outcomes'].items():
            if k:
                aggs += [f'sum({c}_share_{k}) AS {c}_{v}_share_sum',
                         f'sum({c}_component_{k}_spend) AS {c}_{v}_component_spend_sum']
    hy=con.execute('SELECT '+','.join(aggs)+' FROM trips GROUP BY hhid,panel_year ORDER BY household_code,panel_year').df()
    if hy.empty: raise ValueError('No households pass the observed-year filter.')
    hh=hy.household_code.to_numpy(); starts=np.flatnonzero(np.r_[True,hh[1:]!=hh[:-1]])
    ends=np.r_[starts[1:],len(hy)]
    gaps=(hy.groupby('household_code',sort=False).panel_year.diff()>1)
    if YEAR_GAP_POLICY not in ('observed_step','error'): raise ValueError('Invalid YEAR_GAP_POLICY')
    if gaps.any() and YEAR_GAP_POLICY=='error': raise ValueError(f'{gaps.sum()} observed-year gaps; explicit handling required.')
    hy['chain_step']=hy.groupby('household_code',sort=False).cumcount()+1
    x=np.column_stack([np.ones(len(hy)),hy[TRANSITION_EVENT_COLUMNS].to_numpy(float)])
    mask=np.ones(len(hy),bool); mask[starts]=False
    curr=np.flatnonzero(mask); row=np.full(len(hy),-1,dtype=np.int64); row[curr]=np.arange(len(curr))
    annual_cols=['household_code','panel_year','n_trips','food_total_spend_sum',*TRANSITION_EVENT_COLUMNS]
    annual_cols += [c for c in hy if c.endswith(('_n','_spend_sum','_share_sum')) and c not in annual_cols]
    annual=hy[annual_cols].copy()
    base=np.column_stack([np.ones(len(hy)),hy[GATE_COVARIATE_COLS].to_numpy(float)])
    unique,mapping=np.unique(base,axis=0,return_inverse=True)
    blocks={}
    for c,cfg in CATEGORY_CONFIG.items():
        for gate in cfg['gates']:
            success=hy[f'{c}_{cfg["outcomes"][gate["target_code"]]}_share_sum'].to_numpy(float)
            total=sum(hy[f'{c}_{cfg["outcomes"][v]}_share_sum'].to_numpy(float) for v in gate['eligible_codes'])
            active=np.flatnonzero(total>0)
            if not len(active): raise ValueError(f'No purchases for {gate["key"]}')
            blocks[gate['key']]=GateBlock(gate['key'],c,gate['prefix'],unique,success[active]/total[active],active,float(total.sum()),1.,success[active],np.maximum(total[active]-success[active],0.0),mapping[active])
    arrays=ModelArrays({}, {}, {}, np.array([]),np.array([]),{},blocks,np.array([],dtype=int),hy,
        pd.MultiIndex.from_frame(hy[['household_code','panel_year']]),{},hh[starts].tolist(),{},starts,ends,x,curr,row)
    arrays._connection=con
    arrays._annual_statistics=annual
    shell=pd.DataFrame(index=pd.RangeIndex(int(hy.n_trips.sum())))
    shell.attrs['beverage_merge_audit']=merge_audit
    shell.attrs['share_input_audit']=share_audit
    shell.attrs['sorting_audit']={'input_trips':n,'retained_trips':len(shell),'minimum_observed_years_required':MIN_OBSERVED_YEARS_PER_HOUSEHOLD,
        'trips_removed_by_minimum_years_filter':n-len(shell),'observed_year_gaps':int(gaps.sum()),'year_gap_policy':YEAR_GAP_POLICY}
    print(f'Input {n:,} trips; retained {len(shell):,}; {len(starts):,} households; {len(hy):,} household-years; {gaps.sum():,} gaps.')
    prepared = PreparedData(shell,hy,arrays,annual)
    prepared._source_path=str(Path(path).resolve())
    set_household_weights(prepared)
    prepare_incidence_cache(prepared, scratch_dir)
    return prepared


def prepare_incidence_cache(prepared, scratch_dir):
    """Cache exact household x spending counts once; never round spending.

    State-independent incidence factors out of the state recursion. Its full
    refitted log likelihood is added once to the sequence likelihood. A bounded
    in-memory cache avoids all trip scans in bootstrap; large caches stay on disk.
    """
    arrays=prepared.arrays; con=arrays._connection
    path=Path(scratch_dir)/'incidence_household_spend.parquet'
    household_index=pd.DataFrame({'household_code':arrays.households,
                                 'household_index':np.arange(len(arrays.households),dtype=np.int64)})
    con.register('incidence_household_index',household_index)
    fields=['h.household_index','CAST(food_spend AS DOUBLE)/10.0 AS spend10','count(*) AS n']
    fields += [f'sum(CAST({c}_y>0 AS BIGINT)) AS {c}' for c in CATEGORY_NAMES]
    sql='SELECT '+','.join(fields)+' FROM trips JOIN incidence_household_index h ON CAST(trips.hhid AS VARCHAR)=h.household_code GROUP BY 1,2'
    path_sql="'"+str(path).replace("'","''")+"'"
    con.execute('COPY ('+sql+') TO '+path_sql+" (FORMAT PARQUET, COMPRESSION ZSTD)")
    con.unregister('incidence_household_index')
    rows=int(con.execute('SELECT count(*) FROM read_parquet('+path_sql+')').fetchone()[0])
    arrays._incidence_cache_path=str(path.resolve())
    arrays._incidence_cache=None
    # Bound construction peak as well as steady-state storage; no RAM guessing.
    estimated_peak_bytes=rows*160
    if estimated_peak_bytes<=INCIDENCE_CACHE_RAM_MB*1024**2:
        columns=con.execute('SELECT * FROM read_parquet('+path_sql+')').fetchnumpy()
        spend,codes=np.unique(columns.pop('spend10'),return_inverse=True)
        index=columns.pop('household_index')
        arrays._incidence_cache=(spend,codes,index,columns)
        cache_mode='memory'
    else:
        cache_mode='disk'
    arrays._incidence_cache_mode=cache_mode
    prepared.df.attrs['incidence_cache']={'mode':cache_mode,'rows':rows,'memory_limit_mb':INCIDENCE_CACHE_RAM_MB}
    print(f'Exact incidence cache: {rows:,} household-spending groups; {cache_mode} mode.')


def fit_incidence_models(arrays, progress_bar=None):
    """Fully refit all incidence logits from exact household-weighted counts."""
    cache=arrays._incidence_cache
    if cache is not None:
        spend,codes,index,columns=cache
        sample_weight=arrays._hh_weights[index]
        counts={c:np.bincount(codes,weights=columns[c]*sample_weight,minlength=len(spend))
                for c in ['n',*CATEGORY_NAMES]}
        keep=counts['n']>0
        spend=spend[keep];counts={c:values[keep] for c,values in counts.items()}
    else:
        # Each caller gets a private connection, also safe with bootstrap threads.
        con=arrays._connection.cursor()
        try:
            weights=pd.DataFrame({'household_index':np.arange(len(arrays._hh_weights)),
                                  'bootstrap_weight':arrays._hh_weights})
            con.register('bootstrap_weights',weights)
            path_sql="'"+arrays._incidence_cache_path.replace("'","''")+"'"
            fields=['spend10',*[f'sum(d.{c}*w.bootstrap_weight) AS {c}' for c in ['n',*CATEGORY_NAMES]]]
            counts=con.execute('SELECT '+','.join(fields)+' FROM read_parquet('+path_sql+
                ') d JOIN bootstrap_weights w USING(household_index) WHERE bootstrap_weight>0 GROUP BY spend10 ORDER BY spend10').fetchnumpy()
            spend=counts.pop('spend10')
        finally:
            con.close()
    X=np.column_stack([np.ones(len(spend)),spend])
    starts=getattr(arrays,'_incidence_starts',{})
    fits={}
    for c in CATEGORY_NAMES:
        fits[c]=fit_binomial_count_logit(X,counts[c],counts['n']-counts[c],starts.get(c),
            [f'{c}_inc_intercept',f'{c}_inc_food_total_spend_10'],f'incidence:{c}')
    return fits


def initial_gate_weights(prepared):
    """Non-identical emission starts, independent of the initial pi starting value.

    Smoothed annual fresh share supplies a weak initial ordering. Its only role
    is numerical initialization; subsequent posterior weights come from the HMM.
    """
    hy=prepared.hy
    score=(hy.fv_fresh_share_sum.to_numpy(float)+.5)/(hy.fv_share_observed_n.to_numpy(float)+1.)
    z=(score-score.mean())/max(score.std(),1e-8)
    g=expit(.7*np.clip(z,-3,3))
    out=hy[['household_code','panel_year']].copy()
    out['gamma_state_1']=1-g; out['gamma_state_2']=g
    return out


## 3. E-step and safeguarded EM
Forward–backward is vectorized across households with equal sequence lengths. Five choice regressions use posterior-weighted annual fractional masses. Two transition regressions use at most 64 event patterns. Incidence is separable and refitted from cached exact counts once per bootstrap sample, then shared by its two models. Each model estimates its own common pi from first-observed-year pseudo-posteriors, counting each household once. Bootstrap multiplicities enter the household pseudo-likelihood, all M-step counts, and profile aggregates. Warm starts do not fix any parameters. EM convergence is local, not a global-optimum guarantee.


In [ ]:
def _align_gamma_to_hy(gamma_df: pd.DataFrame, arrays: ModelArrays) -> Tuple[np.ndarray, np.ndarray]:
    """Use zero-copy aligned posterior fast path, retaining a safe keyed fallback."""
    if len(gamma_df) == len(arrays.household_year):
        same_hh = np.array_equal(
            gamma_df["household_code"].astype(str).to_numpy(),
            arrays.household_year["household_code"].astype(str).to_numpy(),
        )
        same_year = np.array_equal(
            gamma_df["panel_year"].to_numpy(dtype=np.int64),
            arrays.household_year["panel_year"].to_numpy(dtype=np.int64),
        )
        if same_hh and same_year:
            return (
                gamma_df["gamma_state_1"].to_numpy(dtype=np.float64, copy=False),
                gamma_df["gamma_state_2"].to_numpy(dtype=np.float64, copy=False),
            )
    gamma_key = pd.MultiIndex.from_frame(gamma_df[["household_code", "panel_year"]])
    aligned = gamma_df.set_index(gamma_key).reindex(arrays.hy_key)
    if aligned[["gamma_state_1", "gamma_state_2"]].isna().any().any():
        raise ValueError("gamma_df cannot be aligned to the household-year index.")
    return aligned["gamma_state_1"].to_numpy(dtype=np.float64), aligned["gamma_state_2"].to_numpy(dtype=np.float64)

def fit_gate_models(
    arrays: ModelArrays,
    gamma_df: pd.DataFrame,
    progress_bar: Optional[Any] = None,
    em_iteration: Optional[int] = None,
    max_em_iters: Optional[int] = None,
    previous_fits: Optional[Mapping[str, Any]] = None,
) -> Dict[str, Any]:
    """Fit cached compressed gate likelihoods with one weight pair per eligible row."""
    gamma1, gamma2 = _align_gamma_to_hy(gamma_df, arrays)
    fitted: Dict[str, Any] = {}
    for gate_key in GATE_KEYS:
        block = arrays.gate_blocks[gate_key]
        if progress_bar is not None:
            if em_iteration is None:
                progress_bar.set_description_str(f"Initialization: {progress_bar.n + 1}/{progress_bar.total} | gate={gate_key}")
            else:
                progress_bar.set_description_str(f"EM {em_iteration}/{max_em_iters} | gate={gate_key}")
        # Sum fractional successes/failures over annual groups with the SAME
        # exact design. Four small count vectors replace all trip-state rows.
        n_design = len(block.x_base)
        w1 = gamma1[block.hy_pos]*block.choice_weight*arrays._hy_weights[block.hy_pos]
        w2 = gamma2[block.hy_pos]*block.choice_weight*arrays._hy_weights[block.hy_pos]
        counts = [np.bincount(block.design_pos, weights=w*count, minlength=n_design)
                  for w,count in ((w1,block.success_count), (w1,block.failure_count),
                                  (w2,block.success_count), (w2,block.failure_count))]
        start = previous_fits[gate_key]["coef"] if previous_fits is not None else None
        fitted[gate_key] = fit_gate_count_logit(
            block.x_base, *counts, start=start,
            param_names=[f"{block.prefix}_{col}" for col in GATE_X_COLS],
            block_name=f"gate:{gate_key}",
        )
        fitted[gate_key]["diagnostics"]["compressed_design_rows"] = n_design
        fitted[gate_key]["diagnostics"]["n_observations"] = block.eligible_rows
        if progress_bar is not None and em_iteration is None:
            progress_bar.update(1)
    return fitted

def log_emission_by_household_year(arrays, incidence_models, gate_params):
    """Evaluate the trip fractional pseudo-log-likelihood using annual share sums."""
    n_hy = len(arrays.household_year)
    # Incidence is independent of state, so it cancels in every posterior.
    # Retain its exact weighted LL separately for fit statistics/EM safeguards.
    arrays._incidence_loglik = -sum(float(incidence_models[c]['nll']) for c in CATEGORY_NAMES)
    state1, state2 = np.zeros(n_hy), np.zeros(n_hy)
    for key in GATE_KEYS:
        block = arrays.gate_blocks[key]
        coef = np.asarray(gate_params[key]["coef"], dtype=np.float64)
        shared = block.x_base @ np.r_[coef[0], coef[2:]]
        for state_num, destination in ((1.0,state1), (2.0,state2)):
            eta = shared + state_num * coef[1]
            lp, lq = -np.logaddexp(0, -eta), -np.logaddexp(0, eta)
            ll = (block.success_count*lp[block.design_pos]
                  +block.failure_count*lq[block.design_pos])
            # hy_pos is unique in a gate, so direct indexed addition is valid.
            destination[block.hy_pos] += block.choice_weight*ll
    return np.column_stack([state1,state2])

def forward_backward(arrays: ModelArrays, emission_loglik: np.ndarray, pi_state2: float,
                     beta_from1: np.ndarray, beta_from2: np.ndarray):
    """Same log-space recursions, vectorized over equal-length households.

    Observed-year gaps count as one step only under the explicit observed_step policy.
    """
    n_hy = len(arrays.household_year)
    gamma_values = np.empty((n_hy, 2), dtype=np.float64)
    xi_values = np.empty((len(arrays.transition_curr_pos), 4), dtype=np.float64)
    eta1 = arrays.transition_x @ beta_from1
    eta2 = arrays.transition_x @ beta_from2
    log_trans = np.stack([-np.logaddexp(0, eta1), -np.logaddexp(0, -eta1),
                          -np.logaddexp(0, eta2), -np.logaddexp(0, -eta2)], axis=1).reshape(n_hy, 2, 2)
    log_pi = np.array([safe_log(1-pi_state2), safe_log(pi_state2)])
    if not hasattr(arrays, '_sequence_batches'):
        lengths = arrays.hh_ends - arrays.hh_starts
        arrays._sequence_batches = [(np.flatnonzero(lengths == t),
            arrays.hh_starts[lengths == t, None] + np.arange(t)) for t in np.unique(lengths)]
    household_lls = np.empty(len(arrays.hh_starts))
    for hh_indices, pos in arrays._sequence_batches:
        em = emission_loglik[pos]
        trans = log_trans[pos]
        alpha = np.zeros_like(em); backward = np.zeros_like(em)
        alpha[:, 0] = log_pi + em[:, 0]
        for t in range(1, pos.shape[1]):
            alpha[:, t, 0] = em[:, t, 0]+np.logaddexp(alpha[:, t-1, 0]+trans[:, t, 0, 0], alpha[:, t-1, 1]+trans[:, t, 1, 0])
            alpha[:, t, 1] = em[:, t, 1]+np.logaddexp(alpha[:, t-1, 0]+trans[:, t, 0, 1], alpha[:, t-1, 1]+trans[:, t, 1, 1])
        ll = np.logaddexp(alpha[:, -1, 0], alpha[:, -1, 1])
        household_lls[hh_indices] = ll
        for t in range(pos.shape[1]-2, -1, -1):
            nxt = em[:, t+1]+backward[:, t+1]
            backward[:, t, 0] = np.logaddexp(trans[:, t+1, 0, 0]+nxt[:, 0], trans[:, t+1, 0, 1]+nxt[:, 1])
            backward[:, t, 1] = np.logaddexp(trans[:, t+1, 1, 0]+nxt[:, 0], trans[:, t+1, 1, 1]+nxt[:, 1])
        gamma = np.exp(alpha + backward - ll[:, None, None])
        gamma /= gamma.sum(axis=2, keepdims=True)
        gamma_values[pos] = gamma
        for t in range(1, pos.shape[1]):
            xi = np.exp(alpha[:, t-1, :, None] + trans[:, t] + em[:, t, None, :]
                        + backward[:, t, None, :] - ll[:, None, None])
            xi /= xi.sum(axis=(1, 2), keepdims=True)
            xi_values[arrays.transition_row_of_hy[pos[:, t]]] = xi.reshape(-1, 4)
    # Sum in original household order to minimize numerical changes.
    total_ll = float(arrays._hh_weights @ household_lls) + arrays._incidence_loglik
    gamma_df = arrays.household_year[['household_code', 'panel_year']].copy()
    gamma_df['gamma_state_1'], gamma_df['gamma_state_2'] = gamma_values.T
    gamma_df['decoded_state'] = np.where(gamma_values[:, 1] > gamma_values[:, 0], 2, 1)
    curr = arrays.transition_curr_pos
    xi_df = pd.DataFrame({'household_code': arrays.household_year['household_code'].to_numpy()[curr],
        'year_prev': arrays.household_year['panel_year'].to_numpy()[curr-1],
        'year_curr': arrays.household_year['panel_year'].to_numpy()[curr],
        **{c: xi_values[:, j] for j, c in enumerate(['xi_11', 'xi_12', 'xi_21', 'xi_22'])},
        **{c: arrays.transition_x[curr, j] for j, c in enumerate(TRANSITION_EVENT_COLUMNS, 1)}})
    return total_ll, gamma_df, xi_df

def fit_transition_models(
    xi_df: pd.DataFrame,
    start_from1: Optional[np.ndarray] = None,
    start_from2: Optional[np.ndarray] = None,
    arrays: Optional[ModelArrays] = None,
    maxiter: int = 300,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """Fit both reference-preserving transition logits from compressed counts."""
    # Six binary flags imply at most 64 distinct designs. Cache their mapping.
    if arrays is not None and hasattr(arrays, "_transition_grouping"):
        X, codes = arrays._transition_grouping
    else:
        flags = np.column_stack([xi_df[c].to_numpy(dtype=np.int8) for c in TRANSITION_EVENT_COLUMNS])
        all_codes = flags @ (1 << np.arange(len(TRANSITION_EVENT_COLUMNS)))
        present, codes = np.unique(all_codes, return_inverse=True)
        X = np.column_stack([np.ones(len(present)),
                             ((present[:, None] >> np.arange(len(TRANSITION_EVENT_COLUMNS))) & 1)])
        codes = codes.astype(np.int32)
        if arrays is not None:
            arrays._transition_grouping = X, codes
    counts = {c:np.bincount(codes, weights=xi_df[c].to_numpy()*(arrays._hy_weights[arrays.transition_curr_pos] if arrays is not None else 1.0), minlength=len(X))
              for c in ("xi_11", "xi_12", "xi_21", "xi_22")}
    fit1 = fit_binomial_count_logit(
        X, counts["xi_12"], counts["xi_11"], start_from1,
        [f"trans_from1_{col}" for col in TRANSITION_X_COLS], "transition:from_state_1", maxiter=maxiter,
    )
    fit2 = fit_binomial_count_logit(
        X, counts["xi_22"], counts["xi_21"], start_from2,
        [f"trans_from2_{col}" for col in TRANSITION_X_COLS], "transition:from_state_2", maxiter=maxiter,
    )
    for fit in (fit1,fit2):
        fit["diagnostics"]["compressed_design_rows"] = len(X)
        fit["diagnostics"]["n_observations"] = len(xi_df)
    return fit1, fit2

def _run_monotone_em(prepared, incidence, max_em_iters, tolerance, no_switch=False, start_result=None, verbose=True):
    """Shared EM policy: warm starts, observed-LL guard, persistent plateau.

    A rejected downhill proposal is logged, never relabeled as an improvement.
    Roundoff-sized rejections can count toward a stable plateau only when all
    candidate M-steps satisfy their gradient checks. Larger drops stop with
    converged=False and preserve the last accepted coefficients/posteriors.
    """
    if max_em_iters < 1 or tolerance <= 0:
        raise ValueError('Require max_em_iters >= 1 and tolerance > 0.')
    started_all = time.perf_counter()
    arrays = prepared.arrays
    if not 0 < INITIAL_PI_START < 1:
        raise ValueError('INITIAL_PI_START must be strictly between zero and one.')
    gates = (fit_gate_models(arrays, initial_gate_weights(prepared)) if start_result is None
             else {key: {**fit, 'coef':fit['coef'].copy(), 'table':fit['table'].copy(),
                         'result':copy(fit['result']), 'diagnostics':dict(fit['diagnostics'])}
                   for key,fit in start_result.gate_params.items()})
    pi = INITIAL_PI_START if start_result is None else start_result.pi_state2
    beta1 = np.zeros(len(TRANSITION_X_COLS)); beta2 = beta1.copy()
    if start_result is not None and not no_switch:
        beta1, beta2 = start_result.beta_from1.copy(), start_result.beta_from2.copy()

    def e_step(g, p, b1, b2):
        emission = log_emission_by_household_year(arrays, incidence, g)
        if no_switch:
            ll, gamma = no_switch_e_step(arrays, emission, p)
            return ll, gamma, None
        return forward_backward(arrays, emission, p, b1, b2)

    ll, gamma, xi = e_step(gates, pi, beta1, beta2)
    trans = []
    if not no_switch:
        # Valid coefficient/covariance containers even if the FIRST EM step
        # is rejected. maxiter=0 evaluates the specified zero coefficients.
        trans = list(fit_transition_models(xi, beta1, beta2, arrays, maxiter=0))
    history = []
    stable = 0
    converged = False
    stop_reason = 'max_iterations_reached'
    label = 'No-switch' if no_switch else 'Proposed'
    incidence_ok = all(incidence[c]['result'].success for c in CATEGORY_NAMES)
    for iteration in range(1, max_em_iters + 1):
        started = time.perf_counter()
        first_gamma = gamma.gamma_state_2.to_numpy()[arrays.hh_starts]
        new_pi = float(np.clip(np.average(first_gamma, weights=arrays._hh_weights), PI_EPS, 1-PI_EPS))
        new_trans = [] if no_switch else list(fit_transition_models(xi, beta1, beta2, arrays))
        new_beta1, new_beta2 = (beta1, beta2) if no_switch else tuple(f['coef'] for f in new_trans)
        new_gates = fit_gate_models(arrays, gamma, previous_fits=gates)
        candidate_ll, candidate_gamma, candidate_xi = e_step(new_gates, new_pi, new_beta1, new_beta2)
        candidate_delta = candidate_ll - ll
        failures = sum(not f['result'].success for f in [*new_trans, *new_gates.values()])
        threshold = tolerance + EM_REL_TOL * max(1.0, abs(ll))
        # Tolerance for arithmetic noise, much smaller than an EM stop threshold.
        roundoff = max(1e-8, 64 * np.finfo(float).eps * max(1.0, abs(ll)))
        accepted = bool(np.isfinite(candidate_ll) and candidate_delta >= 0)
        material_drop = not np.isfinite(candidate_ll) or candidate_delta < -roundoff
        delta = 0.0
        if accepted:
            delta = candidate_delta
            pi, gates, trans = new_pi, new_gates, new_trans
            beta1, beta2 = new_beta1, new_beta2
            ll, gamma, xi = candidate_ll, candidate_gamma, candidate_xi
        plateau = not material_drop and delta <= threshold and failures == 0 and incidence_ok
        stable = stable + 1 if plateau else 0
        converged = stable >= EM_STABLE_ITERS
        if material_drop:
            stop_reason = 'downhill_step_rejected'
        elif converged:
            stop_reason = 'stable_likelihood_and_block_gradients'
        history.append({
            'phase': 'EM', 'iteration': iteration, 'log_likelihood': ll, 'pi_state2': pi,
            'log_likelihood_change': delta, 'candidate_log_likelihood_change': candidate_delta,
            'step_accepted': accepted, 'likelihood_parameter_stage': 'after_M_step',
            'iteration_seconds': time.perf_counter()-started,
            'elapsed_seconds': time.perf_counter()-started_all,
            'converged': converged, 'tolerance': tolerance, 'relative_tolerance': EM_REL_TOL,
            'effective_ll_tolerance': threshold, 'stable_iterations': stable,
            'max_em_iters': max_em_iters, 'optimizer_failures': failures,
            'stop_reason': stop_reason,
        })
        if verbose and (iteration == 1 or iteration % EM_PRINT_EVERY == 0 or converged or material_drop or iteration == max_em_iters):
            print(f'{label} EM {iteration:03d}/{max_em_iters}: pseudo_LL={ll:.6f}, '
                  f'delta={delta:.6g}, candidate_delta={candidate_delta:.6g}, '
                  f'block_failures={failures}, stable={stable}/{EM_STABLE_ITERS}')
        if material_drop:
            warnings.warn(f'{label}: downhill EM proposal rejected; last accepted fit retained.', RuntimeWarning)
            break
        if converged:
            break
    return dict(gates=gates, pi=pi, beta1=beta1, beta2=beta2, trans=trans,
                ll=ll, gamma=gamma, history=pd.DataFrame(history), converged=converged,
                stop_reason=stop_reason, estimation_seconds=time.perf_counter()-started_all)

def estimate_model(prepared: PreparedData, max_em_iters: int = MAX_EM_ITERS,
                   tolerance: float = TOL, incidence=None, start_result=None, verbose=True) -> EstimationResult:
    started_all = time.perf_counter()
    if verbose: print('Fitting incidence models; then safeguarded proposed-model EM...')
    if incidence is None: incidence = fit_incidence_models(prepared.arrays)

    out = _run_monotone_em(prepared, incidence, max_em_iters, tolerance, start_result=start_result, verbose=verbose)
    for f in [*[incidence[c] for c in CATEGORY_NAMES], *out['trans'], *out['gates'].values()]:
        finalize_diagnostics(f)
    initial_table = pd.DataFrame([{'parameter':'initial_pi_logit','estimate':np.log(out['pi'])-np.log1p(-out['pi'])}])
    trans1, trans2 = out['trans']
    result = EstimationResult(prepared, incidence, out['gates'], out['beta1'], out['beta2'], out['pi'],
        out['gamma'], out['history'], initial_table,
        pd.concat([trans1['table'], trans2['table']], ignore_index=True), out['ll'],
        {'from_state_1': trans1, 'from_state_2': trans2})
    result.optimizer_diagnostics_df = collect_optimizer_diagnostics(result)
    result.estimation_seconds = time.perf_counter()-started_all
    result.converged = out['converged']
    result.stop_reason = out['stop_reason']
    return result

def collect_optimizer_diagnostics(result: EstimationResult) -> pd.DataFrame:
    """Return one diagnostic row for every final optimized logit block."""
    rows = [result.incidence_models[category]["diagnostics"] for category in CATEGORY_NAMES]
    if result.transition_fits is None:
        raise RuntimeError("Final transition fits are unavailable for optimizer diagnostics.")
    rows.extend([
        result.transition_fits["from_state_1"]["diagnostics"],
        result.transition_fits["from_state_2"]["diagnostics"],
    ])
    rows.extend(result.gate_params[key]["diagnostics"] for key in GATE_KEYS)
    return pd.DataFrame(rows)


In [ ]:
@dataclass
class NoSwitchResult:
    """Two latent household classes with exactly zero switching probabilities."""

    prepared: PreparedData
    incidence_models: Dict[str, Any]
    gate_params: Dict[str, Any]
    pi_state2: float
    gamma_df: pd.DataFrame
    log_likelihood: float
    history_df: pd.DataFrame
    converged: bool
    estimation_seconds: float

def no_switch_e_step(arrays, emission_loglik, pi_state2):
    """Exact household mixture likelihood for P = [[1, 0], [0, 1]].

    Sum annual log emissions within each household BEFORE mixing the two
    classes. Each household has one latent class for its entire sequence.
    Direct mixture evaluation avoids clipping structural zeros to epsilon.
    """
    if not 0 < pi_state2 < 1:
        raise ValueError("Initial class probability must be strictly between 0 and 1.")
    household_emission = np.add.reduceat(emission_loglik, arrays.hh_starts, axis=0)
    joint = household_emission + np.array([np.log1p(-pi_state2), np.log(pi_state2)])
    household_ll = np.logaddexp(joint[:, 0], joint[:, 1])
    posterior = np.exp(joint - household_ll[:, None])
    posterior /= posterior.sum(axis=1, keepdims=True)
    # Contiguous annual groups use the identical posterior in every year.
    gamma = np.repeat(posterior, arrays.hh_ends - arrays.hh_starts, axis=0)
    gamma_df = arrays.household_year[['household_code', 'panel_year']].copy()
    gamma_df['gamma_state_1'], gamma_df['gamma_state_2'] = gamma.T
    gamma_df['decoded_state'] = np.where(gamma[:, 1] > gamma[:, 0], 2, 1)
    return float(arrays._hh_weights @ household_ll) + arrays._incidence_loglik, gamma_df

def estimate_no_switch_model(proposed, max_em_iters=MAX_EM_ITERS, tolerance=TOL, start_result=None, verbose=True):
    """Re-estimate fixed household classes with the same safeguarded optimizer.

    Reuse the exactly separable incidence fits after raw trip buffers are freed.
    The common mixing probability and gate coefficients are independently estimated.
    """
    out = _run_monotone_em(proposed.prepared, proposed.incidence_models,
                           max_em_iters, tolerance, no_switch=True, start_result=start_result, verbose=verbose)
    # Record rank diagnostics; inference is supplied by full bootstrap refits.
    for fit in out['gates'].values():
        finalize_diagnostics(fit)
    result = NoSwitchResult(proposed.prepared, proposed.incidence_models, out['gates'],
        out['pi'], out['gamma'], out['ll'], out['history'], out['converged'], out['estimation_seconds'])
    result.stop_reason = out['stop_reason']
    return result


## 4. Household bootstrap inference and outputs
Both models export every free parameter, bootstrap SE, percentile 95% CI, approximate Wald z/p/stars, and a full bootstrap covariance matrix. `initial_pi_logit` adds one parameter; `initial_probability.csv` reports pi on the probability scale. Proposed pseudo-posterior spending profiles are re-estimated in every accepted draw. Conventional AIC/BIC are unavailable for the fractional objective. Fixed nested transitions have no significance tests. Draw-level estimates and diagnostics are checkpointed atomically after every attempted replicate. Aggregate CSVs refresh every `BOOTSTRAP_EXPORT_EVERY` draws and at completion.

For B accepted household bootstrap refits, all reported variances use
$$\widehat{\mathrm{Var}}_{boot}(\hat\theta)=\frac{1}{B-1}\sum_{b=1}^{B}(\hat\theta^{(b)}-\bar\theta_{boot})^2.$$
SEs are the square roots of these variances. Wald z/p/stars use the original-sample estimate and this bootstrap SE. Percentile confidence intervals use the bootstrap draws directly. No block-wise inverse-Hessian variance is used for inference; Hessians remain only for numerical optimization and rank diagnostics.

**Reporting the method:** We estimate a latent-state fractional-share model by pseudo-likelihood using safeguarded EM and estimate a common initial-state probability separately for each model. We obtain inference using a nonparametric household cluster bootstrap: entire household histories are sampled with replacement, all parameters are re-estimated, and state labels are aligned with the original fit. The empirical covariance of accepted bootstrap estimates supplies standard errors; reported Wald tests use these standard errors and confidence intervals use bootstrap percentiles. Report the number of attempted/successful draws and diagnostics alongside the results. This procedure treats households as the independent sampling units.


In [ ]:
def build_annual_sufficient_statistics(data):
    return data.annual_stats.copy()

def build_transition_matrix(beta_from1: np.ndarray, beta_from2: np.ndarray, x: np.ndarray) -> pd.DataFrame:
    """Build a two-by-two transition matrix at a supplied annual covariate vector."""
    p12 = float(sigmoid(np.dot(beta_from1, x)))
    p22 = float(sigmoid(np.dot(beta_from2, x)))
    return pd.DataFrame({
        "from_state": ["State 1", "State 2"],
        "to_state_1": [1.0 - p12, 1.0 - p22],
        "to_state_2": [p12, p22],
    })

def build_transition_outputs(result: EstimationResult) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Build baseline and event-specific transition matrices at mean covariates."""
    hy = result.prepared.hy
    mean_x = np.array([
        1.0 if col == "intercept" else float(hy[col].mean())
        for col in TRANSITION_X_COLS
    ], dtype=np.float64)
    baseline = build_transition_matrix(result.beta_from1, result.beta_from2, mean_x)
    rows: List[pd.DataFrame] = []
    for event in TRANSITION_EVENT_COLUMNS:
        position = TRANSITION_X_COLS.index(event)
        for event_value in [0.0, 1.0]:
            x = mean_x.copy()
            x[position] = event_value
            matrix = build_transition_matrix(result.beta_from1, result.beta_from2, x)
            matrix.insert(0, "event", event)
            matrix.insert(1, "event_value", int(event_value))
            rows.append(matrix)
    return baseline, pd.concat(rows, ignore_index=True)

def infer_posterior_ratio_difference(household_stats, numerator_stat, denominator_stat):
    """Point estimates only. Uncertainty is filled from complete bootstrap refits."""
    values = []
    for state in (1, 2):
        denominator = float(household_stats[f'{denominator_stat}_s{state}'].sum())
        numerator = float(household_stats[f'{numerator_stat}_s{state}'].sum())
        values.append(numerator/denominator if denominator > 0 else np.nan)
    difference = values[1]-values[0]
    return dict(state_1_estimate=values[0], state_2_estimate=values[1],
        state_1_standard_error=np.nan, state_2_standard_error=np.nan,
        difference_state2_minus_state1=difference, difference_standard_error=np.nan,
        z_value=np.nan, p_value=np.nan, significance_stars='',
        difference_with_stars=f'{difference:.6f}', inference='bootstrap_pending',
        n_household_clusters=len(household_stats))


def _collapse_profile_statistics_to_households(
    annual: pd.DataFrame,
    gamma_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Align the estimation sample and collapse posterior-weighted statistics by household."""
    gamma = gamma_df[[
        "household_code", "panel_year", "gamma_state_1", "gamma_state_2"
    ]].copy()
    if gamma.duplicated(["household_code", "panel_year"]).any():
        raise ValueError("Posterior gamma contains duplicate household-year keys.")

    aligned = annual.merge(
        gamma,
        on=["household_code", "panel_year"],
        how="outer",
        indicator=True,
        validate="one_to_one",
    )
    alignment_summary = pd.DataFrame([{
        "panel_household_years_before_alignment": int(len(annual)),
        "panel_household_years_missing_from_posterior": int((aligned["_merge"] == "left_only").sum()),
        "posterior_household_years_missing_from_panel": int((aligned["_merge"] == "right_only").sum()),
        "matched_household_years_retained": int((aligned["_merge"] == "both").sum()),
    }])
    matched = aligned.loc[aligned["_merge"] == "both"].drop(columns="_merge").reset_index(drop=True)
    if matched.empty:
        raise ValueError("No household-years remain after profile/posterior alignment.")

    excluded = {
        "household_code", "panel_year", "gamma_state_1", "gamma_state_2",
        *TRANSITION_EVENT_COLUMNS,
    }
    stat_columns = [column for column in annual.columns if column not in excluded]
    gamma_state_1 = matched["gamma_state_1"].to_numpy(dtype=np.float64, copy=False)
    gamma_state_2 = matched["gamma_state_2"].to_numpy(dtype=np.float64, copy=False)
    weighted = {"household_code": matched["household_code"]}
    weighted_columns: List[str] = []
    for column in stat_columns:
        values = matched[column].to_numpy(dtype=np.float64, copy=False)
        weighted[f"{column}_s1"] = values * gamma_state_1
        weighted[f"{column}_s2"] = values * gamma_state_2
        weighted_columns.extend([f"{column}_s1", f"{column}_s2"])
    household_stats = (
        pd.DataFrame(weighted)
        .groupby("household_code", sort=False, observed=True)[weighted_columns]
        .sum()
    )
    alignment_summary["matched_households_retained"] = int(len(household_stats))
    return household_stats, alignment_summary

def build_state_profile_summary_with_inference(
    result: EstimationResult,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build five-category weighted profile point estimates for bootstrap inference."""
    annual = result.prepared.annual_stats
    arrays = result.prepared.arrays
    gamma1,gamma2 = _align_gamma_to_hy(result.gamma_df, arrays)
    weight1,weight2 = gamma1*arrays._hy_weights,gamma2*arrays._hy_weights
    excluded={'household_code','panel_year',*TRANSITION_EVENT_COLUMNS}
    totals={}
    for column in annual.columns:
        if column not in excluded:
            values=annual[column].to_numpy(dtype=float,copy=False)
            totals[column+'_s1']=[float(weight1@values)]
            totals[column+'_s2']=[float(weight2@values)]
    household_stats=pd.DataFrame(totals)
    alignment_summary=pd.DataFrame([dict(panel_household_years_before_alignment=len(annual),
        panel_household_years_missing_from_posterior=0,posterior_household_years_missing_from_panel=0,
        matched_household_years_retained=len(annual),matched_households_retained=len(arrays.hh_starts))])
    incidence_rows: List[Dict[str, Any]] = []
    composition_rows: List[Dict[str, Any]] = []
    wallet_rows: List[Dict[str, Any]] = []

    for category, cfg in CATEGORY_CONFIG.items():
        incidence_inference = infer_posterior_ratio_difference(
            household_stats, f"{category}_purchase_n", "n_trips"
        )
        incidence_rows.append({
            "profile_section": "purchase_incidence",
            "category": category,
            "outcome_code": np.nan,
            "outcome_label": "purchase",
            "state_1_purchase_rate": incidence_inference["state_1_estimate"],
            "state_2_purchase_rate": incidence_inference["state_2_estimate"],
            **{key: value for key, value in incidence_inference.items()
               if key not in {"state_1_estimate", "state_2_estimate"}},
        })

        for outcome_code, outcome_label in cfg["outcomes"].items():
            if outcome_code == 0:
                continue
            composition_inference = infer_posterior_ratio_difference(
                household_stats,
                f"{category}_{outcome_label}_share_sum",
                f"{category}_share_observed_n",
            )
            composition_rows.append({
                "profile_section": "within_category_mean_trip_share",
                "category": category,
                "outcome_code": outcome_code,
                "outcome_label": outcome_label,
                "state_1_conditional_share": composition_inference["state_1_estimate"],
                "state_2_conditional_share": composition_inference["state_2_estimate"],
                **{key: value for key, value in composition_inference.items()
                   if key not in {"state_1_estimate", "state_2_estimate"}},
            })

        for outcome_code,outcome_label in cfg['outcomes'].items():
            if not outcome_code: continue
            inference=infer_posterior_ratio_difference(household_stats,
                f'{category}_{outcome_label}_component_spend_sum',f'{category}_spend_sum')
            composition_rows.append(dict(profile_section='within_category_pooled_dollar_share',
                category=category,outcome_code=outcome_code,outcome_label=outcome_label,
                state_1_conditional_share=inference['state_1_estimate'],
                state_2_conditional_share=inference['state_2_estimate'],
                **{k:v for k,v in inference.items() if k not in {'state_1_estimate','state_2_estimate'}}))

        wallet_inference = infer_posterior_ratio_difference(
            household_stats, f"{category}_spend_sum", "food_total_spend_sum"
        )
        wallet_rows.append({
            "profile_section": "wallet_share",
            "category": category,
            "outcome_code": np.nan,
            "outcome_label": "overall_category_wallet_share",
            "state_1_food_wallet_share": wallet_inference["state_1_estimate"],
            "state_2_food_wallet_share": wallet_inference["state_2_estimate"],
            **{key: value for key, value in wallet_inference.items()
               if key not in {"state_1_estimate", "state_2_estimate"}},
        })

    incidence = pd.DataFrame(incidence_rows)
    composition = pd.DataFrame(composition_rows)
    wallet = pd.DataFrame(wallet_rows)
    summary = pd.concat([incidence, composition, wallet], ignore_index=True, sort=False)
    preferred_columns = [
        "profile_section", "category", "outcome_code", "outcome_label",
        "state_1_purchase_rate", "state_2_purchase_rate",
        "state_1_conditional_share", "state_2_conditional_share",
        "state_1_food_wallet_share", "state_2_food_wallet_share",
        "state_1_standard_error", "state_2_standard_error",
        "difference_state2_minus_state1", "difference_standard_error",
        "z_value", "p_value", "significance_stars", "difference_with_stars",
        "inference", "n_household_clusters",
    ]
    summary = summary.reindex(columns=preferred_columns)
    for table in [summary,incidence,composition,wallet]:
        table['n_household_clusters']=int(arrays._hh_weights.sum())
    valid_stars = {"", "*", "**", "***"}
    observed_stars = set(summary["significance_stars"].fillna("").astype(str).unique())
    if not observed_stars.issubset(valid_stars):
        raise AssertionError(f"Unexpected significance-star values: {sorted(observed_stars - valid_stars)}")
    if len(incidence) != len(CATEGORY_CONFIG) or len(wallet) != len(CATEGORY_CONFIG):
        raise AssertionError("State-profile output does not contain all five categories.")
    expected_composition = 2*sum(len(cfg["outcomes"]) - 1 for cfg in CATEGORY_CONFIG.values())
    if len(composition) != expected_composition:
        raise AssertionError("State-profile output does not contain every positive outcome.")
    return summary, incidence, composition, wallet, alignment_summary


In [ ]:
def fitted_blocks(result, proposed):
    blocks=[(f'{c}_incidence',result.incidence_models[c],False) for c in CATEGORY_NAMES]
    if proposed:
        blocks += [(f'transition_{name}',fit,True) for name,fit in result.transition_fits.items()]
    blocks += [(f'choice_{k}',result.gate_params[k],True) for k in GATE_KEYS]
    return blocks


def parameter_table(result, proposed):
    frames=[]
    for name, fit, latent in fitted_blocks(result, proposed):
        frames.append(pd.DataFrame({'equation_name':name, 'parameter':fit['table'].parameter,
                                    'estimate':fit['coef']}))
    frames.append(pd.DataFrame([dict(equation_name='initial_state', parameter='initial_pi_logit',
        estimate=np.log(result.pi_state2)-np.log1p(-result.pi_state2))]))
    table=pd.concat(frames, ignore_index=True)
    assert len(table)==(55 if proposed else 41) and not table.parameter.duplicated().any()
    return table


def coefficient_outputs(result, proposed):
    table=parameter_table(result, proposed)
    draws=getattr(result, 'bootstrap_coefficients', np.empty((0,len(table))))
    valid=getattr(result, 'bootstrap_inference_valid', False)
    table['bootstrap_variance']=np.var(draws, axis=0, ddof=1) if valid else np.nan
    table['standard_error']=np.sqrt(table.bootstrap_variance)
    table['z_statistic']=table.estimate/table.standard_error.where(table.standard_error>0)
    table['p_value']=2*norm.sf(abs(table.z_statistic))
    table['significance_stars']=table.p_value.map(significance_stars)
    table['ci_95_lower']=np.quantile(draws, .025, axis=0) if valid else np.nan
    table['ci_95_upper']=np.quantile(draws, .975, axis=0) if valid else np.nan
    table['estimate_with_stars']=[f'{v:.6f}{star}' for v,star in zip(table.estimate,table.significance_stars)]
    table['covariance_method']='household cluster bootstrap; all model parameters refitted'
    table['ci_method']='percentile bootstrap'
    table['p_value_method']='normal-reference Wald using bootstrap SE; initial_pi_logit null = 0 (pi = 0.5)'
    table['bootstrap_successful_draws']=len(draws)
    table['inference_status']='bootstrap' if valid else 'unavailable: inspect bootstrap diagnostics'
    cov=np.cov(draws,rowvar=False,ddof=1) if valid else np.full((len(table),len(table)),np.nan)
    return table,pd.DataFrame(cov,index=table.parameter,columns=table.parameter)


def save_outputs(proposed,nested,destination):
    """Organized machine-readable and reviewable outputs for both fitted models."""
    root=Path(destination); root.mkdir(parents=True,exist_ok=True)
    comparison=[]
    for name,result,is_proposed in [('proposed_hmm',proposed,True),('nested_no_switch',nested,False)]:
        folder=root/('02_'+name if is_proposed else '03_'+name); folder.mkdir(exist_ok=True)
        coef,cov=coefficient_outputs(result,is_proposed)
        coef.to_csv(folder/'01_full_coefficient_table.csv',index=False)
        cov.to_csv(folder/'02_bootstrap_covariance.csv',index_label='parameter')
        result.gamma_df.rename(columns={'household_code':'hhid'}).to_csv(folder/'06_posterior_household_year.csv',index=False)
        result.history_df.rename(columns={c:c.replace('log_likelihood','log_pseudo_likelihood') for c in result.history_df.columns}).to_csv(folder/'07_estimation_log.csv',index=False)
        diagnostics=pd.DataFrame([f['diagnostics'] for _,f,_ in fitted_blocks(result,is_proposed)])
        diagnostics.to_csv(folder/'09_optimizer_diagnostics.csv',index=False)
        n=len(result.prepared.arrays.hh_starts); ll=result.log_likelihood; k=len(coef)
        row=dict(model=name,log_pseudo_likelihood=ll,n_parameters=k,AIC=np.nan,BIC=np.nan,
            objective_type='latent-state fractional pseudo-likelihood',
            information_criteria_status='unavailable: fractional kernels are not normalized share densities',
            n_households=n,n_household_years=len(result.prepared.hy),n_trips=len(result.prepared.df),
            estimated_initial_pi=result.pi_state2,converged=result.converged,stop_reason=result.stop_reason,
            em_iterations=int(result.history_df.iteration.max()),estimation_seconds=result.estimation_seconds,
            BIC_sample_size_definition='not applicable',
            rank_deficient_blocks=int((diagnostics.hessian_rank<diagnostics.parameter_count).sum()))
        row['bootstrap_successful_draws']=len(getattr(result, 'bootstrap_coefficients', []))
        row['bootstrap_inference_valid']=getattr(result, 'bootstrap_inference_valid', False)
        pd.DataFrame([initial_probability_summary(result)]).to_csv(folder/'initial_probability.csv',index=False)
        comparison.append(row)
        pd.DataFrame([row]).to_csv(folder/'05_model_fit.csv',index=False)
        if is_proposed:
            baseline,events=build_transition_outputs(result)
            baseline['event']='baseline_at_mean_events'
            pd.concat([baseline,events],ignore_index=True).to_csv(folder/'03_transition_matrices_by_event.csv',index=False)
            summary=bootstrap_profile_summary(result)
            summary['signed_difference_with_stars']=[f'{v:+.6f}{s}' if np.isfinite(v) else '' for v,s in zip(summary.difference_state2_minus_state1,summary.significance_stars)]
            summary.to_csv(folder/'04_posterior_purchase_profiles.csv',index=False)
            healthy=summary.loc[(summary.profile_section=='within_category_mean_trip_share') &
                                (summary.outcome_code==1)].copy()
            healthy.to_csv(folder/'11_posterior_healthy_wallet_shares.csv',index=False)
            summary.loc[(summary.profile_section=='within_category_pooled_dollar_share') & (summary.outcome_code==1)].to_csv(folder/'12_posterior_healthy_pooled_dollar_shares.csv',index=False)
            print('\nPseudo-posterior mean within-category healthy spending shares:')
            print(healthy[['category','state_1_conditional_share','state_2_conditional_share',
                           'signed_difference_with_stars']].to_string(index=False))
            fv=summary.loc[(summary.category=='fv')&(summary.outcome_label=='fresh')&(summary.profile_section=='within_category_mean_trip_share')].iloc[0]
            label_ok=bool(fv.difference_state2_minus_state1>0)
            pd.DataFrame([dict(state_2_has_higher_fresh_share=label_ok,
                interpretation='Review all five category profiles before labeling states Healthy/Unhealthy')]).to_csv(folder/'10_state_label_diagnostic.csv',index=False)
            if not label_ok: warnings.warn('State 2 has no higher fresh-FV profile. Inspect initialization/local optimum before using Healthy label.')
        else:
            pd.DataFrame(np.eye(2),index=['from_state_1','from_state_2'],columns=['to_state_1','to_state_2']).to_csv(folder/'03_fixed_transition_matrix.csv')
    comp=pd.DataFrame(comparison)
    for m in ['AIC','BIC']: comp['delta_'+m]=comp[m]-comp[m].min()
    provisional=(not comp.converged.all()) or proposed.log_likelihood<nested.log_likelihood-TOL or comp.rank_deficient_blocks.gt(0).any()
    comp['comparison_status']='provisional: inspect convergence, pseudo-objective and rank diagnostics' if provisional else 'EM criteria met; local optima remain possible'
    folder=root/'01_model_comparison'; folder.mkdir(exist_ok=True)
    comp.to_csv(folder/'nested_vs_proposed.csv',index=False)
    pd.DataFrame([proposed.prepared.df.attrs['sorting_audit']]).to_csv(root/'sample_audit.csv',index=False)
    config=dict(version='V40_share_of_wallet',source_version='attached V39_Final_HMM_FreePi_Bootstrap(1).ipynb',
        cereal_definition='healthy = low plus medium sugar; other = high sugar',
        share_components=SHARE_COMPONENTS,share_denominator='within-category trip expenditure',
        share_weighting='one unit per positive-spend category trip',negative_spend_tolerance=SPENDING_NEGATIVE_TOL,
        objective_type='fractional pseudo-likelihood; no conventional AIC/BIC',
        categories=CATEGORY_NAMES,
        beverage_rule='preferred dollars = healthy beverage plus all milk; denominator adds unhealthy beverage; original merged codes retained for incidence',
        milk_scope='existing ordinary-milk category; no sweetened/flavored milk changes',
        yogurt_definition='unchanged V30 fat_free',
        input_path=str(INPUT_PATH),run_mode=RUN_MODE,transition_covariates=TRANSITION_EVENT_COLUMNS,
        choice_covariates=GATE_COVARIATE_COLS,initial_pi='estimated separately in each model',minimum_years=MIN_OBSERVED_YEARS_PER_HOUSEHOLD,
        bootstrap_reps=BOOTSTRAP_REPS if RUN_MODE!='synthetic' else SYNTHETIC_BOOTSTRAP_REPS,bootstrap_seed=BOOTSTRAP_SEED,
        bootstrap_min_success=BOOTSTRAP_MIN_SUCCESS,bootstrap_min_success_fraction=BOOTSTRAP_MIN_SUCCESS_FRACTION,
        bootstrap_max_em_iters=BOOTSTRAP_MAX_EM_ITERS,bootstrap_boundary_tol=BOOTSTRAP_BOUNDARY_TOL,
        bootstrap_workers=str(BOOTSTRAP_WORKERS),bootstrap_resume=BOOTSTRAP_RESUME,
        bootstrap_export_every=BOOTSTRAP_EXPORT_EVERY,incidence_cache=proposed.prepared.df.attrs['incidence_cache'],
        implementation='V40_fractional_wallet_low_medium_v2',incidence_likelihood_factorized=True,
        gap_policy=YEAR_GAP_POLICY,max_em_iters=MAX_EM_ITERS,absolute_tolerance=TOL,relative_tolerance=EM_REL_TOL,
        numpy_version=np.__version__,pandas_version=pd.__version__,duckdb_version=duckdb.__version__)
    (root/'run_configuration.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    (root/'README.md').write_text("""# V40 share-of-wallet results
Within-category trip expenditure shares; five categories with all existing milk inside beverages.
Cereal healthy numerator = low plus medium sugar; comparison = high sugar.
One cereal gate; 55 proposed / 41 no-switch parameters.
Latent-state fractional pseudo-likelihood, not a normalized continuous-share likelihood.
AIC/BIC deliberately unavailable. Do not compare objectives with V39.
Pseudo-posterior mean trip shares: 11_posterior_healthy_wallet_shares.csv.
Separate pooled-dollar ratios: 12_posterior_healthy_pooled_dollar_shares.csv.
Full profiles distinguish these from category/all-food wallet ratios.
Zero-spend purchases inform incidence only. Inspect share_input_audit.csv.
Household cluster bootstrap refits all parameters, pi and profiles. No Hessian inference.
Percentile intervals; normal-reference Wald p-values. Stars: * p<.10, ** p<.05, *** p<.01.
200 bootstrap draws is a pilot; extend and inspect stability. Synthetic data are software tests only.
Household-year pseudo-posterior weights do not by themselves validate latent-state interpretation.
Convergence is local; sample/gap rules and covariates match V39. See notebook for full specification.
""",encoding='utf-8')
    proposed.prepared.df.attrs['share_input_audit'].to_csv(root/'share_input_audit.csv',index=False)
    proposed.prepared.annual_stats.to_csv(root/'annual_sufficient_statistics.csv',index=False)
    print(comp[['model','log_pseudo_likelihood','n_parameters','converged']].to_string(index=False))
    return comp


In [ ]:
# Household cluster bootstrap. One integer weight per complete household history.
from copy import copy
from types import SimpleNamespace


def swap_state_labels(result, proposed):
    """Exact likelihood-preserving permutation for state_num coded as 1 and 2."""
    for fit in result.gate_params.values():
        beta=fit['coef'].copy()
        beta[0],beta[1]=beta[0]+3*beta[1],-beta[1]
        fit['coef']=beta
        fit['table']['estimate']=beta
        fit['result'].x=beta.copy()
    if proposed:
        beta1,beta2=-result.beta_from2.copy(),-result.beta_from1.copy()
        result.beta_from1,result.beta_from2=beta1,beta2
        for key,beta in [('from_state_1',beta1),('from_state_2',beta2)]:
            fit=result.transition_fits[key]
            fit['coef']=beta
            fit['table']['estimate']=beta
            fit['result'].x=beta.copy()
        result.transition_table=pd.concat([f['table'] for f in result.transition_fits.values()],ignore_index=True)
    result.pi_state2=1-result.pi_state2
    gamma=result.gamma_df
    gamma[['gamma_state_1','gamma_state_2']]=gamma[['gamma_state_2','gamma_state_1']].to_numpy()
    gamma['decoded_state']=3-gamma.decoded_state
    result.history_df['pi_state2']=1-result.history_df.pi_state2
    if proposed:
        result.initial_state_table=pd.DataFrame([{'parameter':'initial_pi_logit',
            'estimate':np.log(result.pi_state2)-np.log1p(-result.pi_state2)}])


def canonicalize_state_labels(result, proposed):
    # Positive fresh-FV state coefficient defines State 2, with all other
    # categories free to have either direction. This does not constrain the fit.
    if result.gate_params['fv_fresh']['coef'][1] < 0:
        swap_state_labels(result, proposed)


def gate_signature(result, reference):
    """Compare labels on the same fixed covariate grid across bootstrap draws."""
    profiles=[]
    for key in GATE_KEYS:
        base=reference.prepared.arrays.gate_blocks[key].x_base
        base=base[np.linspace(0,len(base)-1,min(128,len(base)),dtype=int)]
        beta=result.gate_params[key]['coef']
        shared=base @ np.r_[beta[0],beta[2:]]
        profiles.append(expit(shared[:,None]+np.array([1.,2.])*beta[1]))
    return np.vstack(profiles)


def align_bootstrap_labels(result, reference, proposed):
    target=gate_signature(reference,reference)
    value=gate_signature(result,reference)
    direct=float(np.mean((value-target)**2))
    swapped=float(np.mean((value[:,::-1]-target)**2))
    if swapped < direct:
        swap_state_labels(result,proposed)
    return dict(labels_swapped=swapped<direct,label_match_distance=min(direct,swapped),
                label_match_margin=abs(direct-swapped))


def fit_acceptance(result, proposed):
    fits=[f for _,f,_ in fitted_blocks(result,proposed)]
    issues=[]
    if not result.converged: issues.append('EM_nonconvergence:'+result.stop_reason)
    if not result.history_df.step_accepted.any(): issues.append('no_accepted_EM_updates')
    if not np.isfinite(result.log_likelihood): issues.append('nonfinite_likelihood')
    if not np.isfinite(parameter_table(result,proposed).estimate).all(): issues.append('nonfinite_parameters')
    if any(not f['result'].success for f in fits): issues.append('block_optimizer_failure')
    if any(f['diagnostics']['hessian_rank']<len(f['coef']) for f in fits): issues.append('rank_deficient_block')
    if min(result.pi_state2,1-result.pi_state2)<=BOOTSTRAP_BOUNDARY_TOL: issues.append('pi_at_boundary')
    signature=gate_signature(result,result)
    if np.max(np.abs(signature[:,1]-signature[:,0]))<1e-6: issues.append('collapsed_states')
    return issues


def profile_values(result):
    summary=build_state_profile_summary_with_inference(result)[0]
    state1=summary[['state_1_purchase_rate','state_1_conditional_share','state_1_food_wallet_share']].bfill(axis=1).iloc[:,0]
    state2=summary[['state_2_purchase_rate','state_2_conditional_share','state_2_food_wallet_share']].bfill(axis=1).iloc[:,0]
    return summary, np.column_stack([state1,state2,state2-state1])


def initial_probability_summary(result):
    draws=getattr(result,'bootstrap_coefficients',np.empty((0,1)))
    valid=getattr(result,'bootstrap_inference_valid',False)
    probabilities=expit(draws[:,-1])
    variance=float(np.var(probabilities,ddof=1)) if valid else np.nan
    return dict(parameter='pi_state2',estimate=result.pi_state2,
        bootstrap_variance=variance,standard_error=float(np.sqrt(variance)),
        ci_95_lower=float(np.quantile(probabilities,.025)) if valid else np.nan,
        ci_95_upper=float(np.quantile(probabilities,.975)) if valid else np.nan,
        ci_method='percentile household bootstrap',bootstrap_successful_draws=len(draws),
        inference_status='bootstrap' if valid else 'unavailable: inspect bootstrap diagnostics')


def bootstrap_profile_summary(result):
    summary,points=profile_values(result)
    draws=getattr(result,'bootstrap_profiles',np.empty((0,len(summary),3)))
    valid=getattr(result,'bootstrap_inference_valid',False)
    if valid:
        variance=np.var(draws,axis=0,ddof=1)
        se=np.sqrt(variance)
        low,high=np.quantile(draws,[.025,.975],axis=0)
    else:
        variance=se=low=high=np.full(points.shape,np.nan)
    for j,name in enumerate(['state_1','state_2','difference']):
        summary[name+'_bootstrap_variance']=variance[:,j]
        summary[name+'_standard_error']=se[:,j]
        summary[name+'_ci_95_lower']=low[:,j]
        summary[name+'_ci_95_upper']=high[:,j]
    summary['z_value']=summary.difference_state2_minus_state1/summary.difference_standard_error.where(summary.difference_standard_error>0)
    summary['p_value']=2*norm.sf(abs(summary.z_value))
    summary['significance_stars']=summary.p_value.map(significance_stars)
    summary['difference_with_stars']=[f'{v:.6f}{star}' for v,star in zip(summary.difference_state2_minus_state1,summary.significance_stars)]
    summary['inference']='bootstrap' if valid else 'unavailable: inspect bootstrap diagnostics'
    summary['inference_method']='household cluster bootstrap; refitted posterior weights; percentile CI; normal-reference Wald p'
    summary['bootstrap_successful_draws']=len(draws)
    return summary


# Atomic, deterministic checkpoints; optional shared-memory parallel draws.
import hashlib
import types
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from threadpoolctl import threadpool_limits


def _code_fingerprint(function):
    """Hash executable code without notebook cell IDs or source-line numbers."""
    def describe(value):
        if isinstance(value,types.CodeType):
            return (value.co_code.hex(),value.co_names,value.co_varnames,value.co_argcount,
                    value.co_kwonlyargcount,tuple(describe(v) for v in value.co_consts))
        if isinstance(value,tuple): return tuple(describe(v) for v in value)
        if isinstance(value,frozenset): return tuple(sorted(describe(v) for v in value))
        return repr(value)
    return hashlib.sha256(repr(describe(function.__code__)).encode()).hexdigest()


def bootstrap_manifest(proposed,nested):
    prepared=proposed.prepared
    if not hasattr(prepared,'_source_sha256'):
        digest=hashlib.sha256()
        with open(prepared._source_path,'rb') as handle:
            for block in iter(lambda:handle.read(8*1024**2),b''):digest.update(block)
        prepared._source_sha256=digest.hexdigest()
    functions=[_run_monotone_em,_minimize_logit,fit_gate_count_logit,fit_binomial_count_logit,
        fit_gate_models,fit_transition_models,fit_incidence_models,prepare_incidence_cache,
        log_emission_by_household_year,forward_backward,no_switch_e_step,set_household_weights,
        fit_acceptance,profile_values,build_state_profile_summary_with_inference,
        swap_state_labels,align_bootstrap_labels,gate_signature,
        _finish_logit_fit,_optimizer_diagnostics,_cached_value_gradient,finalize_diagnostics,
        estimate_model,estimate_no_switch_model,_align_gamma_to_hy,parameter_table,
        initial_gate_weights,sigmoid,safe_log,infer_posterior_ratio_difference,
        prepare_parquet,register_combined_beverage_view,register_share_view,run_household_bootstrap]
    config=dict(implementation='V40_fractional_wallet_low_medium_v2',input_sha256=prepared._source_sha256,
        n_households=len(prepared.arrays.hh_starts),n_household_years=len(prepared.hy),
        n_trips=len(prepared.df),seed=BOOTSTRAP_SEED,categories=CATEGORY_CONFIG,
        share_components=SHARE_COMPONENTS,negative_spend_tolerance=SPENDING_NEGATIVE_TOL,
        gate_covariates=GATE_COVARIATE_COLS,transition_covariates=TRANSITION_EVENT_COLUMNS,
        min_years=MIN_OBSERVED_YEARS_PER_HOUSEHOLD,gap_policy=YEAR_GAP_POLICY,
        tolerance=TOL,relative_tolerance=EM_REL_TOL,stable_iterations=EM_STABLE_ITERS,
        opt_ftol=OPT_FTOL,opt_gtol=OPT_GTOL,opt_newton_steps=OPT_NEWTON_STEPS,
        opt_fast_newton_steps=OPT_FAST_NEWTON_STEPS,bootstrap_max_iters=BOOTSTRAP_MAX_EM_ITERS,
        pi_eps=PI_EPS,pi_boundary_tol=BOOTSTRAP_BOUNDARY_TOL,
        numpy_version=np.__version__,pandas_version=pd.__version__,duckdb_version=duckdb.__version__,
        numerical_code={f.__name__:_code_fingerprint(f) for f in functions})
    return dict(config=config,
        original_fit_issues={'proposed_hmm':fit_acceptance(proposed,True),'nested_no_switch':fit_acceptance(nested,False)},
        original_parameters={name:parameter_table(r,flag).estimate.tolist()
            for name,r,flag in [('proposed_hmm',proposed,True),('nested_no_switch',nested,False)]},
        original_likelihoods=[proposed.log_likelihood,nested.log_likelihood])


def _atomic_json(path,data):
    temporary=path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(data,indent=2),encoding='utf-8')
    os.replace(temporary,path)


def run_household_bootstrap(proposed,nested,destination,reps=BOOTSTRAP_REPS,
                            min_success=BOOTSTRAP_MIN_SUCCESS,min_fraction=BOOTSTRAP_MIN_SUCCESS_FRACTION,
                            workers=None):
    """Exact household bootstrap, full refits, reproducible order and safe resume.

    BOOTSTRAP_WORKERS='auto' times two serial draws and two concurrent draws,
    then selects one or two workers. All calibration draws count toward B.
    Workers share read-only caches, own their weights/fits, and use private
    DuckDB connections in disk mode. BLAS threads are capped to prevent nesting.
    No draw is replaced merely because its numerical fit failed.
    """
    if not isinstance(reps,int) or reps<2:raise ValueError('BOOTSTRAP_REPS must be an integer >= 2.')
    if not 0<=min_fraction<=1 or min_success<2:raise ValueError('Invalid bootstrap acceptance threshold.')
    workers=BOOTSTRAP_WORKERS if workers is None else workers
    if workers!='auto' and (not isinstance(workers,int) or workers<1):raise ValueError('Workers must be auto or a positive integer.')
    folder=Path(destination)/'04_bootstrap';folder.mkdir(parents=True,exist_ok=True)
    checkpoint_dir=folder/('checkpoints' if BOOTSTRAP_RESUME else f'checkpoints_{time.time_ns()}')
    checkpoint_dir.mkdir(exist_ok=True)
    manifest=bootstrap_manifest(proposed,nested)
    manifest_path=checkpoint_dir/'manifest.json'
    if manifest_path.exists():
        old=json.loads(manifest_path.read_text())
        same=json.dumps(old['config'],sort_keys=True)==json.dumps(manifest['config'],sort_keys=True)
        same= same and old['original_fit_issues']==manifest['original_fit_issues']
        same= same and all(np.allclose(old['original_parameters'][k],values,rtol=1e-8,atol=1e-6)
            for k,values in manifest['original_parameters'].items())
        same= same and np.allclose(old['original_likelihoods'],manifest['original_likelihoods'],rtol=0,atol=1e-4)
        if not same:
            raise ValueError('Checkpoint dataset/model/settings do not match. Use a new OUTPUT_DIR or set BOOTSTRAP_RESUME=False; existing checkpoints are preserved.')
    else:_atomic_json(manifest_path,manifest)
    references={'proposed_hmm':(proposed,True),'nested_no_switch':(nested,False)}
    names={name:parameter_table(ref,flag).parameter.tolist() for name,(ref,flag) in references.items()}
    base_issues={name:fit_acceptance(ref,flag) for name,(ref,flag) in references.items()}
    for name,issues in base_issues.items():
        if issues:warnings.warn(f'{name}: original fit unsuitable for bootstrap inference: {issues}')
    metadata,_=profile_values(proposed)
    metadata=metadata[['profile_section','category','outcome_code','outcome_label']].copy()
    H=len(proposed.prepared.arrays.hh_starts)
    seeds={b:int(seed.generate_state(1,dtype=np.uint64)[0])
           for b,seed in enumerate(np.random.SeedSequence(BOOTSTRAP_SEED).spawn(reps),1)}
    required=max(min_success,int(math.ceil(min_fraction*reps)))
    if required>reps:warnings.warn(f'Need {required} successful draws; only {reps} requested. Inference will be withheld.')
    completed={}
    for b in range(1,reps+1):
        path=checkpoint_dir/f'replicate_{b:06d}.npz'
        if not path.exists():continue
        try:
            with np.load(path,allow_pickle=False) as saved:
                records=json.loads(str(saved['records'].item()))
                if len(records)!=2 or any(r['replicate']!=b or int(r['draw_seed'])!=seeds[b] for r in records):
                    raise ValueError('Invalid checkpoint seed/replicate.')
                payload={'records':records}
                for record in records:
                    name=record['model']
                    if record['accepted']:
                        values=saved[name].copy()
                        if values.shape!=(len(names[name]),) or not np.isfinite(values).all():raise ValueError('Invalid coefficient checkpoint.')
                        payload[name]=values
                        if name=='proposed_hmm':
                            profiles=saved['profiles'].copy()
                            if profiles.shape!=(len(metadata),3) or not np.isfinite(profiles).all():raise ValueError('Invalid profile checkpoint.')
                            payload['profiles']=profiles
                completed[b]=payload
        except Exception as exc:
            quarantine=path.with_suffix(f'.corrupt_{time.time_ns()}')
            path.rename(quarantine)
            warnings.warn(f'Checkpoint {b} could not be verified ({exc}); retained as {quarantine.name}; rerunning that same draw.')
    if completed:print(f'Resuming: {len(completed)}/{reps} completed draws loaded; no refitting of those draws.',flush=True)

    def draw_one(b):
        started=time.perf_counter()
        rng=np.random.default_rng(seeds[b]);multiplicity=np.bincount(rng.integers(0,H,size=H),minlength=H)
        prepared=copy(proposed.prepared);prepared.arrays=copy(proposed.prepared.arrays)
        set_household_weights(prepared,multiplicity)
        prepared.arrays._incidence_starts={c:proposed.incidence_models[c]['coef'] for c in CATEGORY_NAMES}
        shared_error='';incidence=None;payload={};records=[]
        if not all(base_issues.values()):
            try:
                incidence=fit_incidence_models(prepared.arrays)
                for fit in incidence.values():finalize_diagnostics(fit)
            except Exception as exc:shared_error=f'{type(exc).__name__}: {exc}'
        for name,(reference,flag) in references.items():
            record=dict(replicate=b,draw_seed=str(seeds[b]),model=name,
                unique_sampled_households=int((multiplicity>0).sum()),sampled_households=int(multiplicity.sum()),
                accepted=False,labels_swapped=False,log_pseudo_likelihood=np.nan,pi_state2=np.nan,
                converged=False,error='',status='failed')
            try:
                if base_issues[name]:raise RuntimeError('Original fit invalid: '+'; '.join(base_issues[name]))
                if shared_error:raise RuntimeError('Incidence refit failed: '+shared_error)
                if flag:
                    result=estimate_model(prepared,max_em_iters=BOOTSTRAP_MAX_EM_ITERS,
                        incidence=incidence,start_result=reference,verbose=False)
                else:
                    result=estimate_no_switch_model(SimpleNamespace(prepared=prepared,incidence_models=incidence),
                        max_em_iters=BOOTSTRAP_MAX_EM_ITERS,start_result=reference,verbose=False)
                record.update(align_bootstrap_labels(result,reference,flag))
                record.update(log_pseudo_likelihood=result.log_likelihood,pi_state2=result.pi_state2,
                    converged=result.converged,em_iterations=len(result.history_df))
                issues=fit_acceptance(result,flag)
                if record['label_match_margin']<1e-10:issues.append('ambiguous_state_matching')
                table=parameter_table(result,flag);assert table.parameter.tolist()==names[name]
                if flag:
                    _,values=profile_values(result)
                    if not np.isfinite(values).all():issues.append('nonfinite_profile')
                if issues:record.update(status='excluded',error='; '.join(issues))
                else:
                    record.update(accepted=True,status='accepted');payload[name]=table.estimate.to_numpy(copy=True)
                    if flag:payload['profiles']=values
            except Exception as exc:record['error']=f'{type(exc).__name__}: {exc}'
            record['elapsed_seconds']=time.perf_counter()-started;records.append(record)
        payload['records']=records
        target=checkpoint_dir/f'replicate_{b:06d}.npz';temporary=target.with_suffix('.npz.tmp')
        with open(temporary,'wb') as handle:
            np.savez_compressed(handle,records=np.array(json.dumps(records)),
                **{k:v for k,v in payload.items() if k!='records'})
        os.replace(temporary,target)
        return b,payload

    def export_aggregates():
        records=[record for b in sorted(completed) for record in completed[b]['records']]
        pd.DataFrame(records).to_csv(folder/'replicate_diagnostics.csv',index=False)
        for name in references:
            ids=[b for b in sorted(completed) if name in completed[b]]
            table=pd.DataFrame([completed[b][name] for b in ids],columns=names[name])
            table.insert(0,'replicate',ids);table.to_csv(folder/(name+'_coefficient_draws.csv'),index=False)
        frames=[]
        for b in sorted(completed):
            if 'profiles' in completed[b]:
                frame=metadata.copy();frame.insert(0,'replicate',b)
                frame[['state_1_estimate','state_2_estimate','difference']]=completed[b]['profiles'];frames.append(frame)
        table=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(columns=['replicate',*metadata.columns,'state_1_estimate','state_2_estimate','difference'])
        table.to_csv(folder/'proposed_profile_draws.csv',index=False)

    def accept(answer):
        b,payload=answer;completed[b]=payload
        successes=', '.join(f'{name}: {sum(name in v for v in completed.values())}' for name in references)
        print(f'Bootstrap {b} saved ({len(completed)}/{reps} complete); accepted {successes}',flush=True)
        if len(completed)%BOOTSTRAP_EXPORT_EVERY==0:export_aggregates()

    def run_batch(indices,count):
        if count==1:
            for b in indices:accept(draw_one(b))
            return
        executor=ThreadPoolExecutor(max_workers=count)
        iterator=iter(indices);pending={}
        try:
            for _ in range(count):
                b=next(iterator,None)
                if b is not None:pending[executor.submit(draw_one,b)]=b
            while pending:
                done,_=wait(pending,return_when=FIRST_COMPLETED)
                for future in done:
                    pending.pop(future);accept(future.result())
                    b=next(iterator,None)
                    if b is not None:pending[executor.submit(draw_one,b)]=b
        finally:
            executor.shutdown(wait=True,cancel_futures=True)

    remaining=[b for b in range(1,reps+1) if b not in completed]
    timing={};actual_workers=1
    # Threadpoolctl is scoped around the whole pool; no thread changes global
    # BLAS limits independently of other active workers.
    with threadpool_limits(limits=1,user_api='blas'):
        if workers=='auto' and len(remaining)>=4 and (os.cpu_count() or 1)>=2:
            start=time.perf_counter();run_batch(remaining[:2],1);serial=time.perf_counter()-start
            start=time.perf_counter();run_batch(remaining[2:4],2);parallel=time.perf_counter()-start
            actual_workers=2 if parallel<.90*serial else 1
            timing=dict(serial_two_draws_seconds=serial,parallel_two_draws_seconds=parallel)
            remaining=remaining[4:]
            print(f'Automatic worker calibration selected {actual_workers} worker(s).',flush=True)
        elif workers!='auto':actual_workers=workers
        run_batch(remaining,actual_workers)
    export_aggregates()
    rows=[]
    for name,(reference,flag) in references.items():
        ids=[b for b in sorted(completed) if name in completed[b]]
        reference.bootstrap_coefficients=np.asarray([completed[b][name] for b in ids],dtype=float).reshape(-1,len(names[name]))
        reference.bootstrap_inference_valid=bool(len(ids)>=required and not base_issues[name])
        if flag:reference.bootstrap_profiles=np.asarray([completed[b]['profiles'] for b in ids],dtype=float).reshape(-1,len(metadata),3)
        rows.append(dict(model=name,requested_draws=reps,successful_draws=len(ids),excluded_or_failed=reps-len(ids),
            minimum_successes_required=required,inference_valid=reference.bootstrap_inference_valid,
            original_fit_issues='; '.join(base_issues[name]),seed=BOOTSTRAP_SEED,
            method='nonparametric household cluster bootstrap',synthetic_smoke_test=RUN_MODE=='synthetic',
            worker_setting=str(workers),selected_workers=actual_workers,checkpoint_directory=str(checkpoint_dir),
            incidence_cache_mode=proposed.prepared.arrays._incidence_cache_mode,**timing))
        if not reference.bootstrap_inference_valid:warnings.warn(f'{name}: bootstrap inference withheld; inspect 04_bootstrap diagnostics.')
    summary=pd.DataFrame(rows);summary.to_csv(folder/'bootstrap_summary.csv',index=False)
    return summary


## 5. Execute and resume
Use the same original `INPUT_PATH` and Run All. Results go to `V40_Share_of_Wallet_LowMediumHealthy_Results`; synthetic mode uses a separate folder and five test draws.

- `BOOTSTRAP_REPS=200` is the pilot setting. For final reporting, increase to 1,000 and check Monte Carlo stability. Existing compatible draws are reused.
- `BOOTSTRAP_WORKERS='auto'` chooses between one and two workers using measured throughput. Explicit worker counts are also supported.
- `INCIDENCE_CACHE_RAM_MB=256` bounds cache construction; larger caches use disk. Reserve disk space in the output location.
- `BOOTSTRAP_RESUME=True` verifies and reuses atomic checkpoints in `04_bootstrap/checkpoints`. Restart Kernel and Run All after interruption or when increasing B. Original fits are recomputed and checked against the manifest before old draws can be reused.
- A different dataset, specification, numerical implementation or estimation setting requires a fresh checkpoint run. Use a new `OUTPUT_DIR` or `BOOTSTRAP_RESUME=False` to preserve old checkpoints and start afresh. Failed fits are logged as completed attempts; they are never silently replaced with new random draws.

Inspect convergence, bootstrap success counts, rank/boundary diagnostics and state profiles before using results. Aggregate CSVs are updated periodically and at completion; per-draw checkpoints are saved immediately.


In [ ]:
def make_synthetic_parquet(path, n_households=120, seed=SEED):
    """Artificial fixture with all schema fields and both states; never research data."""
    rng=np.random.default_rng(seed); rows=[]; trip=0
    for h in range(n_households):
        state=int(rng.random()<.3)
        for t in range(4):
            events=rng.binomial(1,.15,6)
            if t: state=int(rng.random()<(.8 if state else .2))
            controls=[rng.binomial(1,.5),rng.uniform(1,12),rng.integers(1,6),rng.uniform(25,75)]
            for j in range(12):
                trip+=1
                row=dict(hhid=f'{h:06d}',panel_year=2008+t,trip_id=str(trip),purchase_date=pd.Timestamp(2008+t,1,1)+pd.Timedelta(days=j*20),food_spend=float(rng.integers(10,100)),initial_fresh_fv_share_2007=.3,other_unemployment=0)
                row.update(zip(TRANSITION_EVENT_COLUMNS,events)); row.update(zip(GATE_COVARIATE_COLS,controls))
                for c,cfg in CATEGORY_CONFIG.items():
                    for v in cfg['spending']: row[v]=0.
                    row[c+'_y']=0
                    if rng.random()<.7:
                        groups=SHARE_COMPONENTS[c] if c!='beverage' else [['healthy_beverage_spending'],['unhealthy_beverage_spending']]
                        m=len(groups); p=.85 if state else .15
                        probs=np.r_[p,np.repeat((1-p)/(m-1),m-1)]
                        if rng.random()<.35:
                            q=np.eye(m)[rng.choice(m,p=probs)]
                        else:
                            q=rng.dirichlet(8*probs)
                        total=rng.uniform(.1,15)
                        for group,share in zip(groups,q): row[group[0]]=float(total*share)
                        row[c+'_y']=int(np.flatnonzero(q>0)[0]+1)
                        if c=='cereal':
                            # Write the ORIGINAL three-level source schema; the read-only view merges it.
                            low=float(row['low_sugar_cereal_spending'])*float(rng.random())
                            row['medium_sugar_cereal_spending']=row['low_sugar_cereal_spending']-low
                            row['low_sugar_cereal_spending']=low
                            row['cereal_y']=1 if low>0 else (2 if row['medium_sugar_cereal_spending']>0 else 3)
                row['milk_y']=int(rng.choice([0,1,2],p=[.65,.15,.20]))
                milk_columns=['skimmed_milk_spending','low_fat_milk_spending','other_milk_spending']
                if row['milk_y']:
                    column=milk_columns[0] if row['milk_y']==1 else milk_columns[int(rng.integers(1,3))]
                    row[column]=float(rng.uniform(.1,10))
                row['food_spend']=sum(row[v] for cfg in CATEGORY_CONFIG.values() for v in cfg['spending'])+float(rng.uniform(1,30))
                rows.append(row)
    pd.DataFrame(rows).to_parquet(path,index=False)


started=time.perf_counter()
if RUN_MODE not in ('full_estimation','validate_only','synthetic'): raise ValueError('Unknown RUN_MODE')
if RUN_MODE=='synthetic':
    destination=Path.cwd()/'V40_Share_of_Wallet_LowMediumHealthy_Synthetic_Results'; destination.mkdir(exist_ok=True)
    source=destination/'synthetic_input.parquet'
    make_synthetic_parquet(source)
    iterations=SYNTHETIC_EM_ITERS
else:
    source=INPUT_PATH; destination=OUTPUT_DIR; iterations=MAX_EM_ITERS
    if not source.is_file(): raise FileNotFoundError(source)
prepared=prepare_parquet(source,destination/'duckdb_temp')
print(f'Preparation: {time.perf_counter()-started:.1f} seconds')
destination.mkdir(parents=True,exist_ok=True)
merge_audit=prepared.df.attrs['beverage_merge_audit']
merge_audit.to_csv(destination/'beverage_merge_audit.csv',index=False)
prepared.df.attrs['share_input_audit'].to_csv(destination/'share_input_audit.csv',index=False)
print('Share construction audit:')
print(prepared.df.attrs['share_input_audit'].to_string(index=False))
print('Retained-trip milk + beverage merge audit:')
print(merge_audit.to_string(index=False))
if RUN_MODE=='validate_only':
    prepared.arrays._connection.close()
    destination.mkdir(parents=True,exist_ok=True)
    pd.DataFrame([prepared.df.attrs['sorting_audit']]).to_csv(destination/'sample_audit.csv',index=False)
else:
    proposed=estimate_model(prepared,max_em_iters=iterations)
    nested=estimate_no_switch_model(proposed,max_em_iters=iterations)
    canonicalize_state_labels(proposed, True)
    canonicalize_state_labels(nested, False)
    try:
        run_household_bootstrap(proposed, nested, destination,
            reps=SYNTHETIC_BOOTSTRAP_REPS if RUN_MODE=='synthetic' else BOOTSTRAP_REPS,
            min_success=2 if RUN_MODE=='synthetic' else BOOTSTRAP_MIN_SUCCESS,
            min_fraction=0.0 if RUN_MODE=='synthetic' else BOOTSTRAP_MIN_SUCCESS_FRACTION)
        comparison=save_outputs(proposed,nested,destination)
    finally:
        prepared.arrays._connection.close()
    assert 0 < proposed.pi_state2 < 1 and 0 < nested.pi_state2 < 1
    print(f'Total elapsed: {time.perf_counter()-started:.1f} seconds. Results: {destination.resolve()}')
